# LeetCode 4073 — Count Good Strings (Hard)

Notebook này dạy bài **Count Good Strings** (Weekly Contest 522, câu 4, độ khó Hard, rating ≈ 2125) **từ con số 0**.

Skeleton trên LeetCode (tên hàm suy ra từ đường dẫn `count-good-strings` theo quy ước camelCase của LeetCode — hãy đối chiếu với editor khi nộp):

```python
class Solution:
    def countGoodStrings(self, n: int) -> int:
        ...
```

Điểm "khó" của bài này nằm ở hai chỗ:

- Đề cho một **định nghĩa đệ quy** (chuỗi tốt được ghép từ các chuỗi tốt nhỏ hơn) nhìn rất rối.
- `n` lên tới `10^15` — không thể duyệt chuỗi, thậm chí **không thể chạy một vòng lặp tới `n`**.

Ta sẽ đi từng bước: hiểu định nghĩa bằng tay → viết brute force bám sát đề → làm thí nghiệm để tìm **quy luật ẩn** → đếm bằng quy hoạch động → tăng tốc tới `O(log n)` bằng lũy thừa ma trận và fast doubling.

---

## Sau notebook này, bạn sẽ nắm được

1. Cách đọc một **định nghĩa đệ quy** (recursive definition) và viết brute force bám sát nó, dùng `lru_cache` để nhớ kết quả.
2. Khái niệm **run** (đoạn dài nhất gồm các ký tự giống nhau liên tiếp) và cách lấy run bằng `itertools.groupby`.
3. Kỹ năng **"thí nghiệm với n nhỏ → đoán quy luật → chứng minh"**, biến định nghĩa đệ quy thành điều kiện đơn giản: *mọi run có độ dài lẻ*.
4. Cách đếm bằng **quy hoạch động** (Dynamic Programming, DP): DP theo "phần cuối", rồi DP theo **trạng thái** (automaton).
5. Nhận ra dãy **Fibonacci** và chứng minh nó bằng hai cách (đại số và song ánh).
6. Kỹ thuật **lũy thừa nhanh** (exponentiation by squaring) cho số và cho **ma trận 2×2**, cùng số học **modulo `10^9 + 7`**.
7. Công thức **fast doubling** cho Fibonacci và bản LeetCode chạy `O(log n)` với `n = 10^15`.

## Các cách giải sẽ học (từ dễ đến tối ưu)

| Cách | Ý tưởng | Thời gian | Dùng được tới khoảng |
|---|---|---|---|
| Cách 1 | Brute force: sinh mọi chuỗi, kiểm tra đúng theo định nghĩa đệ quy | `O(2^n · n^2)` | `n ≈ 15` |
| Cách 2 | DP đếm số cách chia `n` thành các số lẻ (theo phần cuối) | `O(n^2)` | `n ≈ 10^3 – 10^4` |
| Cách 3 | DP trạng thái: run hiện tại dài chẵn hay lẻ (2 biến) | `O(n)` | `n ≈ 10^7` |
| Cách 4 | Lũy thừa ma trận 2×2 | `O(log n)` | `n = 10^15` ✓ |
| Cách 5 | Fast doubling Fibonacci | `O(log n)` | `n = 10^15` ✓ (gọn và nhanh nhất) |

# 1. Đề bài

Cho một số nguyên `n`.

Một chuỗi được gọi là **tốt** (good) nếu nó **chỉ gồm các ký tự `'a'` và `'b'`**, và thỏa **một trong hai** điều kiện sau:

1. **(Luật cơ sở)** Chuỗi chứa **đúng một loại ký tự** và có **độ dài lẻ**.
   Ví dụ: `"a"`, `"bbb"`, `"aaaaa"`.
2. **(Luật ghép)** Chuỗi viết được dưới dạng `s = s1 + s2`, trong đó `s1` và `s2` là hai chuỗi tốt **khác rỗng**, và **ký tự cuối của `s1` khác ký tự đầu của `s2`**.

Hãy trả về **số chuỗi tốt có độ dài `n`**, lấy **modulo `10^9 + 7`**.

Ở đây `+` là phép **nối chuỗi** (concatenation): `"ab" + "ba" = "abba"`.

---

## Ví dụ

**Ví dụ 1:** `n = 4` → `6`

- Các chuỗi tốt: `"aaab"`, `"abbb"`, `"baaa"`, `"bbba"`, `"abab"`, `"baba"`.
- `"aaab" = "aaa" + "b"`: hai phần đều tốt (một loại ký tự, độ dài lẻ) và ở chỗ nối `'a' ≠ 'b'`.
- `"ab" = "a" + "b"` là tốt, nên `"abab" = "ab" + "ab"` cũng tốt vì ở chỗ nối `'b' ≠ 'a'`.

**Ví dụ 2:** `n = 3` → `4`. Các chuỗi tốt: `"aaa"`, `"bbb"`, `"aba"`, `"bab"`.

**Ví dụ 3:** `n = 2` → `2`. Các chuỗi tốt: `"ab"`, `"ba"`.

## Ràng buộc

```text
1 <= n <= 10^15
```

In [1]:
MOD = 10**9 + 7

# (n, đáp án mong đợi, danh sách chuỗi tốt mà đề liệt kê)
examples = [
    (4, 6, ["aaab", "abbb", "baaa", "bbba", "abab", "baba"]),
    (3, 4, ["aaa", "bbb", "aba", "bab"]),
    (2, 2, ["ab", "ba"]),
]

for n, expected, good_list in examples:
    print(f"n = {n} -> expected = {expected}, chuỗi tốt theo đề: {good_list}")
    assert len(good_list) == expected

print("MOD =", MOD)
print("n lớn nhất =", 10**15)

n = 4 -> expected = 6, chuỗi tốt theo đề: ['aaab', 'abbb', 'baaa', 'bbba', 'abab', 'baba']
n = 3 -> expected = 4, chuỗi tốt theo đề: ['aaa', 'bbb', 'aba', 'bab']
n = 2 -> expected = 2, chuỗi tốt theo đề: ['ab', 'ba']
MOD = 1000000007
n lớn nhất = 1000000000000000


# 2. Ràng buộc nói gì về độ phức tạp?

`n` có thể bằng `10^15` (một triệu tỷ). Python thuần làm được khoảng `10^7` phép tính đơn giản mỗi giây (ước lượng thô).

| Độ phức tạp | Số phép khi `n = 10^15` | Kết luận |
|---|---|---|
| `O(2^n)` | `2^(10^15)` — một số có khoảng `3·10^14` chữ số | không tưởng |
| `O(n^2)` | `10^30` | không |
| `O(n)` | `10^15` ≈ vài năm chạy liên tục | không |
| `O(log n)` | `log2(10^15) ≈ 50` | tức thì ✓ |

Vậy ta **bắt buộc** phải tìm cách `O(log n)` (hoặc `O(log^2 n)`, ...).

> Dấu hiệu kinh điển: **`n` khổng lồ + "modulo 10^9 + 7"** ⇒ đáp án thường là một **dãy truy hồi tuyến tính** và ta tính nó bằng **lũy thừa nhanh / lũy thừa ma trận**.

Cũng để ý: đề bắt lấy modulo vì số chuỗi tốt là một số **cực lớn**, không thể in ra nguyên vẹn.

In [2]:
import math

n_max = 10**15
print("log2(10^15) ≈", round(math.log2(n_max), 2))
print("bit_length của 10^15 =", n_max.bit_length())

digits_2_pow_n = int(n_max * math.log10(2)) + 1
print(f"2^(10^15) có khoảng {digits_2_pow_n:.3e} chữ số")

ops_per_sec = 10**7
years = n_max / ops_per_sec / (3600 * 24 * 365)
print("Một vòng lặp O(n) với 10^7 phép/giây mất khoảng", round(years, 1), "năm")

log2(10^15) ≈ 49.83
bit_length của 10^15 = 50
2^(10^15) có khoảng 3.010e+14 chữ số
Một vòng lặp O(n) với 10^7 phép/giây mất khoảng 3.2 năm


# 3. Hiểu đề bằng tay (chưa cần thuật toán)

Trước khi code, ta tự kiểm tra vài chuỗi nhỏ để hiểu **hai luật** hoạt động thế nào.

## n = 1

- `"a"`: một loại ký tự, độ dài 1 (lẻ) → **tốt** theo luật cơ sở.
- `"b"`: tương tự → **tốt**.

Luật ghép cần `s1`, `s2` **khác rỗng**, nên chuỗi dài 1 không ghép được. Kết quả: **2** chuỗi tốt.

## n = 2

| Chuỗi | Luật cơ sở? | Luật ghép (cách cắt duy nhất `s[:1] + s[1:]`) | Tốt? |
|---|---|---|---|
| `aa` | không (độ dài 2 chẵn) | `"a" + "a"`: hai phần tốt, nhưng chỗ nối `'a' = 'a'` ✗ | ❌ |
| `ab` | không (2 loại ký tự) | `"a" + "b"`: hai phần tốt, `'a' ≠ 'b'` ✓ | ✅ |
| `ba` | không | `"b" + "a"` ✓ | ✅ |
| `bb` | không | `"b" + "b"`: chỗ nối `'b' = 'b'` ✗ | ❌ |

Kết quả **2**, khớp Ví dụ 3.

> **Bài học đầu tiên:** luật ghép **cấm** dán hai khối cùng ký tự cạnh nhau. Vì vậy `"aa"` **không thể** sinh ra từ `"a" + "a"`.

## n = 3 — xét đủ 8 chuỗi

Chuỗi dài 3 có hai cách cắt: `s[:1] + s[1:]` và `s[:2] + s[2:]`.

| Chuỗi | Luật cơ sở? | Thử cắt | Tốt? |
|---|---|---|---|
| `aaa` | có (một loại, dài 3) | — | ✅ |
| `aab` | không | `"a"+"ab"`: nối `a`=`a` ✗ ; `"aa"+"b"`: `"aa"` không tốt ✗ | ❌ |
| `aba` | không | `"a"+"ba"`: `"ba"` tốt, nối `a`≠`b` ✓ | ✅ |
| `abb` | không | `"a"+"bb"`: `"bb"` không tốt ✗ ; `"ab"+"b"`: nối `b`=`b` ✗ | ❌ |
| `baa` | không | `"b"+"aa"` ✗ ; `"ba"+"a"`: nối `a`=`a` ✗ | ❌ |
| `bab` | không | `"b"+"ab"` ✓ | ✅ |
| `bba` | không | `"b"+"ba"`: nối `b`=`b` ✗ ; `"bb"+"a"` ✗ | ❌ |
| `bbb` | có | — | ✅ |

Kết quả **4**, khớp Ví dụ 2.

## n = 4 — vì sao `"abba"` không tốt?

Thử cả 3 cách cắt:

- `"a" + "bba"`: theo bảng n = 3, `"bba"` không tốt ✗
- `"ab" + "ba"`: chỗ nối `'b' = 'b'` ✗
- `"abb" + "a"`: `"abb"` không tốt ✗

⇒ `"abba"` **không tốt**. Tương tự `"aabb"`, `"aaaa"` cũng không tốt. Sáu chuỗi tốt là `aaab, abbb, baaa, bbba, abab, baba`.

## Câu hỏi gợi mở

Hãy nhìn **độ dài các khối ký tự giống nhau liên tiếp**:

| Chuỗi | Các khối | Độ dài khối | Tốt? |
|---|---|---|---|
| `aaab` | `aaa`, `b` | 3, 1 | ✅ |
| `abab` | `a`, `b`, `a`, `b` | 1, 1, 1, 1 | ✅ |
| `aabb` | `aa`, `bb` | 2, 2 | ❌ |
| `abba` | `a`, `bb`, `a` | 1, 2, 1 | ❌ |
| `aaaa` | `aaaa` | 4 | ❌ |

Bạn có thấy quy luật không? Ta sẽ kiểm chứng bằng code ở **Bước đột phá 1**.

# 4. Nền tảng Python và thuật toán

Phần này giới thiệu **mọi công cụ** mà các lời giải phía sau dùng tới. Mỗi công cụ: *nó là gì → ví dụ nhỏ → cell code → liên hệ với bài*.

## 4.1 Chuỗi: nối, cắt (slicing) và kiểm tra luật cơ sở

- `s1 + s2` — nối hai chuỗi: `"ab" + "ab" = "abab"`.
- `s[:k]` — lấy `k` ký tự **đầu**; `s[k:]` — lấy **phần còn lại** từ vị trí `k`.
- Muốn cả hai phần **khác rỗng** thì `k` chạy từ `1` tới `len(s) - 1`, tức `range(1, len(s))`.
- Ký tự **cuối** của `s1 = s[:k]` là `s[k - 1]`; ký tự **đầu** của `s2 = s[k:]` là `s[k]`.
- `set(s)` — tập các ký tự **khác nhau** trong `s`. `len(set(s)) == 1` nghĩa là "chỉ có một loại ký tự".
- `len(s) % 2 == 1` nghĩa là "độ dài lẻ" (`%` là phép chia lấy dư).

**Liên hệ với bài:** luật cơ sở = `len(set(s)) == 1 and len(s) % 2 == 1`; luật ghép = thử mọi `k` và so sánh `s[k-1]` với `s[k]`.

In [3]:
s = "abab"
print("s =", s, "| len =", len(s))
for k in range(1, len(s)):          # k = 1, 2, 3 -> hai phần đều khác rỗng
    left, right = s[:k], s[k:]
    print(f"k={k}: s1={left!r:6} s2={right!r:6} cuối s1={s[k-1]!r} đầu s2={s[k]!r} -> khác nhau? {s[k-1] != s[k]}")

print()
for t in ["a", "aaa", "aa", "ab"]:
    one_kind = len(set(t)) == 1
    odd_len = len(t) % 2 == 1
    print(f"{t!r:6} set={sorted(set(t))} một loại? {one_kind!s:5} lẻ? {odd_len!s:5} -> luật cơ sở: {one_kind and odd_len}")

s = abab | len = 4
k=1: s1='a'    s2='bab'  cuối s1='a' đầu s2='b' -> khác nhau? True
k=2: s1='ab'   s2='ab'   cuối s1='b' đầu s2='a' -> khác nhau? True
k=3: s1='aba'  s2='b'    cuối s1='a' đầu s2='b' -> khác nhau? True

'a'    set=['a'] một loại? True  lẻ? True  -> luật cơ sở: True
'aaa'  set=['a'] một loại? True  lẻ? True  -> luật cơ sở: True
'aa'   set=['a'] một loại? True  lẻ? False -> luật cơ sở: False
'ab'   set=['a', 'b'] một loại? False lẻ? False -> luật cơ sở: False


## 4.2 Sinh mọi chuỗi bằng `itertools.product`

`product("ab", repeat=n)` sinh **mọi bộ `n` ký tự**, mỗi ký tự lấy từ `"ab"`. Mỗi bộ là một `tuple` như `('a', 'b', 'a')`; dùng `"".join(...)` để ghép thành chuỗi `"aba"`.

Mỗi vị trí có 2 lựa chọn nên có đúng **`2^n` chuỗi**.

**Liên hệ với bài:** brute force phải duyệt hết `2^n` chuỗi này — đó là lý do nó chỉ dùng được với `n` nhỏ.

In [4]:
from itertools import product

all_3 = ["".join(t) for t in product("ab", repeat=3)]
print(all_3)

for n in range(1, 6):
    print(f"n = {n}: có {sum(1 for _ in product('ab', repeat=n))} chuỗi = 2^{n} = {2**n}")

['aaa', 'aab', 'aba', 'abb', 'baa', 'bab', 'bba', 'bbb']
n = 1: có 2 chuỗi = 2^1 = 2
n = 2: có 4 chuỗi = 2^2 = 4
n = 3: có 8 chuỗi = 2^3 = 8
n = 4: có 16 chuỗi = 2^4 = 16
n = 5: có 32 chuỗi = 2^5 = 32


## 4.3 Đệ quy và memoization (`functools.lru_cache`)

**Đệ quy** (recursion): hàm gọi lại chính nó với bài toán **nhỏ hơn**, và phải có **trường hợp cơ sở** để dừng.

Định nghĩa "chuỗi tốt" là đệ quy: muốn biết `s` tốt không, ta hỏi `s1`, `s2` (ngắn hơn) có tốt không. Vì vậy hàm kiểm tra cũng sẽ đệ quy.

**Vấn đề:** cùng một chuỗi con bị hỏi đi hỏi lại rất nhiều lần.

**Memoization** (ghi nhớ): lưu kết quả của mỗi tham số đã tính; lần sau gặp lại thì trả luôn. Trong Python chỉ cần đặt `@lru_cache(maxsize=None)` (từ `functools`) phía trên hàm.

Ví dụ kinh điển: Fibonacci `fib(n) = fib(n-1) + fib(n-2)`.

In [5]:
from functools import lru_cache

calls_plain = 0
def fib_plain(n):
    global calls_plain
    calls_plain += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

calls_memo = 0
@lru_cache(maxsize=None)
def fib_memo(n):
    global calls_memo
    calls_memo += 1
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

print("fib_plain(20) =", fib_plain(20), "| số lần gọi hàm:", calls_plain)
print("fib_memo(20)  =", fib_memo(20), "| số lần tính thật:", calls_memo)
print(fib_memo.cache_info())

fib_plain(20) = 6765 | số lần gọi hàm: 21891
fib_memo(20)  = 6765 | số lần tính thật: 21
CacheInfo(hits=18, misses=21, maxsize=None, currsize=21)


Không có memo: **21891** lần gọi. Có memo: chỉ **21** lần tính thật (mỗi giá trị `0..20` một lần), các lần còn lại là "hit" trong cache.

## 4.4 Run — đoạn chạy tối đa, và `itertools.groupby`

**Run** (maximal run) là một đoạn **liên tiếp** gồm **cùng một ký tự** mà **không thể kéo dài thêm** sang trái hay phải.

Ví dụ `"aaabbaaab"` có 4 run: `"aaa"`, `"bb"`, `"aaa"`, `"b"` với độ dài `3, 2, 3, 1`.

Tính chất quan trọng: **hai run kề nhau luôn khác ký tự** (nếu giống thì chúng đã là một run).

`groupby(s)` gom các phần tử **liên tiếp bằng nhau** thành nhóm, trả về từng cặp `(ký tự, nhóm)`. `len(list(nhóm))` là độ dài run.

**Liên hệ với bài:** ta sẽ chứng minh "chuỗi tốt" ⇔ "mọi run có độ dài lẻ".

In [6]:
from itertools import groupby

def runs(s):
    """Trả về list (ký tự, độ dài) của các run trong s."""
    return [(ch, len(list(group))) for ch, group in groupby(s)]

print(runs("aaabbaaab"))
print(runs("abab"))
print(runs("aaaa"))
print("Độ dài run của 'abba':", [length for _, length in runs("abba")])

[('a', 3), ('b', 2), ('a', 3), ('b', 1)]
[('a', 1), ('b', 1), ('a', 1), ('b', 1)]
[('a', 4)]
Độ dài run của 'abba': [1, 2, 1]


## 4.5 Số học modulo `10^9 + 7`

Đáp án của bài là một số khổng lồ, nên đề yêu cầu trả về **phần dư khi chia cho `MOD = 10^9 + 7`**.

Hai tính chất cho phép **lấy mod sau mỗi bước** mà không làm sai kết quả cuối:

```text
(a + b) % m == ((a % m) + (b % m)) % m
(a * b) % m == ((a % m) * (b % m)) % m
```

Nhờ vậy mọi số trung gian luôn nhỏ hơn `m` (khoảng 30 bit), phép tính nhanh.

Chú ý thêm:

- Trong Python, `%` với `m > 0` **luôn trả về số không âm**: `(-3) % 7 = 4`. Trong C++/Java, `-3 % 7 = -3` (phải tự cộng thêm `m`).
- Số nguyên Python **không bao giờ tràn** (big integer), nhưng số càng lớn thì tính càng chậm.

In [7]:
MOD = 10**9 + 7
a, b = 123456789123, 987654321987
print((a + b) % MOD == ((a % MOD) + (b % MOD)) % MOD)
print((a * b) % MOD == ((a % MOD) * (b % MOD)) % MOD)
print("(-3) % 7 =", (-3) % 7)

# Fibonacci thứ 300: không lấy mod vs lấy mod từng bước
x, y = 0, 1
for _ in range(300):
    x, y = y, x + y
print("F(300) có", len(str(x)), "chữ số")

x2, y2 = 0, 1
for _ in range(300):
    x2, y2 = y2, (x2 + y2) % MOD
print("F(300) % MOD tính ở cuối     :", x % MOD)
print("F(300) % MOD lấy mod từng bước:", x2)

True
True
(-3) % 7 = 4
F(300) có 63 chữ số
F(300) % MOD tính ở cuối     : 644264086
F(300) % MOD lấy mod từng bước: 644264086


## 4.6 Biểu diễn nhị phân và phép toán bit

- `bin(13)` → `'0b1101'`: `13 = 8 + 4 + 1`.
- `e & 1` → **bit cuối** của `e` (bằng 1 nếu `e` lẻ).
- `e >> 1` → **bỏ bit cuối** (giống `e // 2`).
- `e.bit_length()` → số bit của `e`.

**Liên hệ với bài:** lũy thừa nhanh duyệt từng bit của số mũ. `10^15` chỉ có **50 bit**, nên vòng lặp chỉ chạy 50 lần.

In [8]:
e = 13
print("bin(13) =", bin(e))
while e > 0:
    print(f"e = {e:2d} ({bin(e)[2:]:>4}) | bit cuối e & 1 = {e & 1} | e >> 1 = {e >> 1}")
    e >>= 1

print("(10**15).bit_length() =", (10**15).bit_length())
print("số bit 1 trong 10**15 =", bin(10**15).count("1"))

bin(13) = 0b1101
e = 13 (1101) | bit cuối e & 1 = 1 | e >> 1 = 6
e =  6 ( 110) | bit cuối e & 1 = 0 | e >> 1 = 3
e =  3 (  11) | bit cuối e & 1 = 1 | e >> 1 = 1
e =  1 (   1) | bit cuối e & 1 = 1 | e >> 1 = 0
(10**15).bit_length() = 50
số bit 1 trong 10**15 = 20


## 4.7 Lũy thừa nhanh (exponentiation by squaring)

Muốn tính `a^e`. Cách ngây thơ: nhân `a` với chính nó `e - 1` lần → `O(e)`.

Ý tưởng: viết `e` ở hệ nhị phân. Ví dụ `13 = 1101₂ = 8 + 4 + 1`, nên

```text
3^13 = 3^8 · 3^4 · 3^1
```

Ta lần lượt **bình phương** `base`: `3, 3^2, 3^4, 3^8, ...` (mỗi lần một phép nhân). Bit nào bằng 1 thì nhân `base` hiện tại vào `result`.

Số vòng lặp = số bit của `e` ≈ `log2(e)`.

**Bất biến (invariant):** trước mỗi vòng, `result · base^e` luôn bằng `a^(e ban đầu)`. Khi `e = 0` thì `result` chính là đáp án.

Python có sẵn `pow(a, e, m)` làm đúng việc này. **Liên hệ với bài:** Cách 4 dùng y hệt, chỉ thay "nhân số" bằng "nhân ma trận".

In [9]:
def fast_pow(a, e, m):
    result = 1
    base = a % m
    steps = 0
    while e > 0:
        if e & 1:                    # bit cuối = 1 -> nhân base vào result
            result = result * base % m
        base = base * base % m       # base: a^1 -> a^2 -> a^4 -> a^8 ...
        e >>= 1
        steps += 1
    return result, steps

print(fast_pow(3, 13, 10**9), "| 3**13 =", 3**13)
value, steps = fast_pow(2, 10**15, MOD)
print("2^(10^15) % MOD =", value, "| số vòng lặp:", steps)
print("Kiểm tra với pow có sẵn:", pow(2, 10**15, MOD))

(1594323, 4) | 3**13 = 1594323
2^(10^15) % MOD = 264444359 | số vòng lặp: 50
Kiểm tra với pow có sẵn: 264444359


## 4.8 Nhân ma trận 2×2

Ma trận 2×2 là bảng 2 hàng, 2 cột. Phép nhân theo quy tắc **"hàng nhân cột"**:

```text
[[p, q],     [[w, x],     [[p·w + q·y,  p·x + q·z],
 [r, s]]  ·   [y, z]]  =   [r·w + s·y,  r·x + s·z]]
```

Nhân ma trận với một **vector** `[u, v]`:

```text
[[p, q],  · [u, v] = [p·u + q·v,  r·u + s·v]
 [r, s]]
```

Đây chính là **"một bước cập nhật hai biến"** — sẽ dùng ở Cách 4.

- Ma trận đơn vị `I = [[1, 0], [0, 1]]` đóng vai trò số 1: `A · I = A`. Và `M^0 = I`.
- Phép nhân ma trận **có tính kết hợp** `(A·B)·C = A·(B·C)`, nên lũy thừa nhanh áp dụng được.

Cell dưới tính `M^1 ... M^6` với `M = [[1, 1], [1, 0]]` — hãy để ý các con số xuất hiện.

In [10]:
def mat_mul_plain(A, B):
    """Nhân hai ma trận 2x2 (chưa lấy mod)."""
    return [
        [A[0][0] * B[0][0] + A[0][1] * B[1][0], A[0][0] * B[0][1] + A[0][1] * B[1][1]],
        [A[1][0] * B[0][0] + A[1][1] * B[1][0], A[1][0] * B[0][1] + A[1][1] * B[1][1]],
    ]

M = [[1, 1], [1, 0]]
P = [[1, 0], [0, 1]]      # ma trận đơn vị = M^0
for k in range(1, 7):
    P = mat_mul_plain(P, M)
    print(f"M^{k} = {P}")

M^1 = [[1, 1], [1, 0]]
M^2 = [[2, 1], [1, 1]]
M^3 = [[3, 2], [2, 1]]
M^4 = [[5, 3], [3, 2]]
M^5 = [[8, 5], [5, 3]]
M^6 = [[13, 8], [8, 5]]


Các phần tử là `1, 2, 3, 5, 8, 13` — dãy **Fibonacci**! Ta sẽ hiểu vì sao ở Cách 4.

## 4.9 Quy hoạch động (DP) và dãy Fibonacci

**Quy hoạch động** (Dynamic Programming): giải bài lớn từ các bài nhỏ, **lưu kết quả bài nhỏ** để không tính lại. Một lời giải DP cần 3 thứ:

1. **Trạng thái**: `dp[i]` nghĩa là gì (phải nói được bằng lời).
2. **Công thức chuyển**: tính `dp[i]` từ các `dp` nhỏ hơn.
3. **Giá trị khởi đầu**.

Dãy Fibonacci là DP đơn giản nhất:

```text
F(0) = 0, F(1) = 1, F(i) = F(i-1) + F(i-2)
0, 1, 1, 2, 3, 5, 8, 13, 21, ...
```

**Liên hệ với bài:** ta sẽ phát hiện đáp án = `2 · F(n)`.

In [11]:
def fib_list(count):
    """count số Fibonacci đầu tiên: F(0), F(1), ..., F(count-1)."""
    F = [0, 1]
    while len(F) < count:
        F.append(F[-1] + F[-2])
    return F[:count]

print(fib_list(15))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]


# 5. Cách 1 — Brute force bám sát định nghĩa

Đây là cách **người mới nghĩ ra đầu tiên** và cũng là cách **hiển nhiên đúng** vì nó dịch nguyên văn đề bài.

## Ý tưởng

1. Sinh tất cả `2^n` chuỗi độ dài `n` gồm `'a'`, `'b'`.
2. Với mỗi chuỗi, kiểm tra "tốt" bằng hàm đệ quy `is_good(s)`:
   - Nếu luật cơ sở đúng → `True`.
   - Thử **mọi** vị trí cắt `k = 1 .. len(s) - 1`: nếu `s[k-1] != s[k]` **và** `is_good(s[:k])` **và** `is_good(s[k:])` → `True`.
   - Không cách cắt nào được → `False`.
3. Đếm số chuỗi tốt.

## Vì sao đúng?

Thử mọi `k` nghĩa là thử **mọi cách viết** `s = s1 + s2` với hai phần khác rỗng — đúng nguyên văn luật ghép. Hai phần ngắn hơn `s` nên đệ quy chắc chắn dừng.

## Mẹo nhỏ

- Kiểm tra `s[k-1] != s[k]` **trước** (rất rẻ). Toán tử `and` của Python dừng ngay khi gặp `False` (short-circuit), nên tránh được nhiều lời gọi đệ quy vô ích.
- `@lru_cache` để mỗi chuỗi con chỉ bị kiểm tra một lần.

In [12]:
from functools import lru_cache
from itertools import product

MOD = 10**9 + 7

@lru_cache(maxsize=None)
def is_good_by_definition(s: str) -> bool:
    # Luật 1: đúng một loại ký tự và độ dài lẻ
    if len(set(s)) == 1 and len(s) % 2 == 1:
        return True
    # Luật 2: thử mọi cách cắt s = s1 + s2 (hai phần khác rỗng)
    for k in range(1, len(s)):
        if s[k - 1] != s[k] and is_good_by_definition(s[:k]) and is_good_by_definition(s[k:]):
            return True
    return False


def count_good_strings_bruteforce(n: int) -> int:
    count = 0
    for chars in product("ab", repeat=n):
        if is_good_by_definition("".join(chars)):
            count += 1
    return count % MOD


for n, expected, _ in examples:
    result = count_good_strings_bruteforce(n)
    print(f"n = {n}: brute force = {result}, expected = {expected}")
    assert result == expected
print("Brute force khớp cả 3 ví dụ!")

n = 4: brute force = 6, expected = 6
n = 3: brute force = 4, expected = 4
n = 2: brute force = 2, expected = 2
Brute force khớp cả 3 ví dụ!


In [13]:
def list_good_strings(n: int):
    """Liệt kê mọi chuỗi tốt độ dài n (theo định nghĩa gốc)."""
    return ["".join(t) for t in product("ab", repeat=n) if is_good_by_definition("".join(t))]

print("n = 4:", list_good_strings(4))
print("n = 5:", list_good_strings(5))
print("Số chuỗi tốt n = 1..12:", [count_good_strings_bruteforce(n) for n in range(1, 13)])

n = 4: ['aaab', 'abab', 'abbb', 'baaa', 'baba', 'bbba']
n = 5: ['aaaaa', 'aaaba', 'abaaa', 'ababa', 'abbba', 'baaab', 'babab', 'babbb', 'bbbab', 'bbbbb']
Số chuỗi tốt n = 1..12: [2, 2, 4, 6, 10, 16, 26, 42, 68, 110, 178, 288]


## Độ phức tạp của Cách 1

- Phải duyệt **`2^n` chuỗi**.
- Cache dùng chung cho mọi chuỗi, nên tổng số chuỗi con khác nhau từng được kiểm tra không quá `2^1 + 2^2 + ... + 2^n < 2^(n+1)`. Mỗi chuỗi con dài `L` tốn `O(L^2)` (thử `L` vị trí cắt, mỗi lần cắt chuỗi tốn `O(L)`).
- Tổng: **Time `O(2^n · n^2)`**, **Space `O(2^n · n)`** (cache lưu các chuỗi).

Với `n = 40` đã có hơn một nghìn tỷ chuỗi; với `n = 10^15` thì hoàn toàn vô vọng.

> Vai trò của Cách 1: làm **"thước đo chuẩn" (oracle)** — mọi cách nhanh phía sau đều phải cho cùng kết quả với nó trên `n` nhỏ.

In [14]:
info = is_good_by_definition.cache_info()
print("Số chuỗi khác nhau đã lưu trong cache:", info.currsize)
print("Giới hạn trên 2^1 + ... + 2^12 =", sum(2**L for L in range(1, 13)))
for n in [20, 40, 60]:
    print(f"n = {n}: phải duyệt 2^{n} = {2**n:,} chuỗi")

Số chuỗi khác nhau đã lưu trong cache: 8190
Giới hạn trên 2^1 + ... + 2^12 = 8190
n = 20: phải duyệt 2^20 = 1,048,576 chuỗi
n = 40: phải duyệt 2^40 = 1,099,511,627,776 chuỗi
n = 60: phải duyệt 2^60 = 1,152,921,504,606,846,976 chuỗi


# 6. Bước đột phá 1 — Chuỗi tốt "trông" như thế nào?

Brute force cho ta một "máy sinh ví dụ". Hãy dùng nó làm **thí nghiệm**: in tất cả chuỗi tốt với `n = 1..5` kèm **độ dài các run** (Mục 4.4).

In [15]:
def run_lengths(s):
    """Danh sách độ dài các run của s (s có thể là chuỗi hoặc tuple)."""
    return [len(list(g)) for _, g in groupby(s)]

for n in range(1, 6):
    print(f"--- n = {n} ---")
    for s in list_good_strings(n):
        print(f"  {s:6} runs = {run_lengths(s)}")

--- n = 1 ---
  a      runs = [1]
  b      runs = [1]
--- n = 2 ---
  ab     runs = [1, 1]
  ba     runs = [1, 1]
--- n = 3 ---
  aaa    runs = [3]
  aba    runs = [1, 1, 1]
  bab    runs = [1, 1, 1]
  bbb    runs = [3]
--- n = 4 ---
  aaab   runs = [3, 1]
  abab   runs = [1, 1, 1, 1]
  abbb   runs = [1, 3]
  baaa   runs = [1, 3]
  baba   runs = [1, 1, 1, 1]
  bbba   runs = [3, 1]
--- n = 5 ---
  aaaaa  runs = [5]
  aaaba  runs = [3, 1, 1]
  abaaa  runs = [1, 1, 3]
  ababa  runs = [1, 1, 1, 1, 1]
  abbba  runs = [1, 3, 1]
  baaab  runs = [1, 3, 1]
  babab  runs = [1, 1, 1, 1, 1]
  babbb  runs = [1, 1, 3]
  bbbab  runs = [3, 1, 1]
  bbbbb  runs = [5]


## Quan sát

**Mọi độ dài run đều là số lẻ!** Ví dụ `aaab → [3, 1]`, `ababa → [1, 1, 1, 1, 1]`, `abbba → [1, 3, 1]`.

Còn các chuỗi xấu ở Mục 3: `aabb → [2, 2]`, `abba → [1, 2, 1]`, `aaaa → [4]` — đều có ít nhất một run **chẵn**.

## Giả thuyết

> **Chuỗi `s` tốt ⇔ mọi run của `s` có độ dài lẻ.**

Một giả thuyết chỉ dựa trên vài ví dụ thì chưa đủ. Bước 1: kiểm tra trên **tất cả** chuỗi với `n ≤ 12` (8190 chuỗi), so với định nghĩa gốc. Bước 2: chứng minh.

In [16]:
def is_good_runs(s: str) -> bool:
    return all(length % 2 == 1 for length in run_lengths(s))

checked = 0
mismatch = []
for n in range(1, 13):
    for t in product("ab", repeat=n):
        s = "".join(t)
        checked += 1
        if is_good_runs(s) != is_good_by_definition(s):
            mismatch.append(s)

print("Đã kiểm tra", checked, "chuỗi; số chuỗi lệch:", len(mismatch))
assert not mismatch

for s in ["aabb", "abba", "aaaa", "aaab", "abab", "aaabaaa"]:
    print(f"{s:8} runs = {run_lengths(s)} -> tốt? {is_good_runs(s)}")

Đã kiểm tra 8190 chuỗi; số chuỗi lệch: 0
aabb     runs = [2, 2] -> tốt? False
abba     runs = [1, 2, 1] -> tốt? False
aaaa     runs = [4] -> tốt? False
aaab     runs = [3, 1] -> tốt? True
abab     runs = [1, 1, 1, 1] -> tốt? True
aaabaaa  runs = [3, 1, 3] -> tốt? True


## Chứng minh giả thuyết (hai chiều)

### Chiều (⇐): mọi run lẻ ⇒ tốt

Quy nạp theo **số run** `r`.

- `r = 1`: chuỗi chỉ là một run độ dài lẻ → tốt theo **luật cơ sở**.
- `r ≥ 2`: tách `s1` = **run đầu tiên**, `s2` = phần còn lại.
  - `s1` là một run lẻ → tốt (luật cơ sở).
  - `s2` có `r - 1` run, đều lẻ → tốt (giả thiết quy nạp).
  - Ký tự cuối của `s1` khác ký tự đầu của `s2` vì **hai run kề nhau luôn khác ký tự**.
  - ⇒ `s` tốt theo **luật ghép**.

### Chiều (⇒): tốt ⇒ mọi run lẻ

Quy nạp theo **độ dài**.

- Nếu `s` tốt nhờ luật cơ sở: `s` là một run duy nhất, độ dài lẻ ✓.
- Nếu `s = s1 + s2` theo luật ghép: `s1`, `s2` ngắn hơn và tốt, nên (quy nạp) mọi run của chúng đều lẻ. Vì **cuối `s1` ≠ đầu `s2`**, run cuối của `s1` và run đầu của `s2` **không dính vào nhau**. Do đó danh sách run của `s` chính là run của `s1` nối với run của `s2` → đều lẻ ✓.

### Trực giác

Luật ghép chỉ dán các khối **tại chỗ đổi ký tự**, nên **không bao giờ làm một khối dài thêm**. Mỗi khối giữ nguyên độ dài lẻ từ lúc sinh ra (luật cơ sở). Một khối chẵn như `"aa"` không bao giờ xuất hiện vì `"a" + "a"` bị cấm.

> Đây chính là phần "Hard" của bài: một định nghĩa đệ quy phức tạp hóa ra chỉ là **một điều kiện cục bộ rất đơn giản**.

# 7. Bước đột phá 2 — Chuỗi tốt ↔ (ký tự đầu, cách chia `n` thành các số lẻ)

Một chuỗi `'a'/'b'` được xác định **hoàn toàn** bởi:

1. **Ký tự đầu tiên** (`'a'` hoặc `'b'`) — vì các run **luân phiên** ký tự: a, b, a, b, ...
2. **Dãy độ dài các run** `(l1, l2, ..., lr)` với `l1 + l2 + ... + lr = n`.

Chuỗi tốt ⇔ mọi `li` lẻ. Một dãy số nguyên dương **có thứ tự** có tổng bằng `n` gọi là một **composition** của `n` (khác với "phân hoạch" partition: ở đây `1 + 3` và `3 + 1` là **hai** cách khác nhau).

Vậy:

```text
Số chuỗi tốt độ dài n = 2 × C(n)
C(n) = số composition của n thành các phần LẺ
```

## Ví dụ n = 4

Composition lẻ của 4: `1+3`, `3+1`, `1+1+1+1` → `C(4) = 3` → đáp án `2 × 3 = 6` ✓

| Composition | Bắt đầu bằng `'a'` | Bắt đầu bằng `'b'` |
|---|---|---|
| `1 + 3` | `abbb` | `baaa` |
| `3 + 1` | `aaab` | `bbba` |
| `1 + 1 + 1 + 1` | `abab` | `baba` |

Cell dưới sinh composition bằng **đệ quy với `yield`** (generator: hàm "nhả" dần từng kết quả thay vì trả cả danh sách một lần), rồi dựng lại chuỗi.

In [17]:
def odd_compositions(n):
    """Sinh mọi cách viết n = l1 + l2 + ... (có thứ tự) với mọi li lẻ."""
    if n == 0:
        yield []                       # composition rỗng của 0
        return
    for first in range(1, n + 1, 2):   # phần đầu: 1, 3, 5, ...
        for rest in odd_compositions(n - first):
            yield [first] + rest

def build_string(first_char, parts):
    """Dựng chuỗi từ ký tự đầu và độ dài các run (luân phiên ký tự)."""
    other = "b" if first_char == "a" else "a"
    pieces = []
    for idx, length in enumerate(parts):
        ch = first_char if idx % 2 == 0 else other
        pieces.append(ch * length)
    return "".join(pieces)

comps = list(odd_compositions(4))
print("Composition lẻ của 4:", comps)
built = sorted(build_string(c, p) for p in comps for c in "ab")
print("Chuỗi dựng được:", built)
print("Khớp brute force?", built == sorted(list_good_strings(4)))
print("C(n) với n = 1..10:", [len(list(odd_compositions(n))) for n in range(1, 11)])

Composition lẻ của 4: [[1, 1, 1, 1], [1, 3], [3, 1]]
Chuỗi dựng được: ['aaab', 'abab', 'abbb', 'baaa', 'baba', 'bbba']
Khớp brute force? True
C(n) với n = 1..10: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


# 8. Cách 2 — DP đếm composition lẻ, `O(n^2)`

Liệt kê composition vẫn tốn thời gian theo cấp số mũ. Ta chỉ cần **đếm** → dùng DP.

## Trạng thái

`dp[i]` = số composition lẻ của `i`.

`dp[0] = 1`: số 0 có đúng **một** cách viết — "không chọn phần nào" (composition rỗng). Quy ước này cần để công thức đúng khi phần cuối dài đúng bằng `i`.

## Công thức chuyển: nhìn vào **phần cuối cùng**

Phần cuối có độ dài `k` lẻ (`k = 1, 3, 5, ... ≤ i`). Bỏ phần cuối đi, phần còn lại là một composition lẻ của `i - k`. Vậy:

```text
dp[i] = dp[i-1] + dp[i-3] + dp[i-5] + ...
```

## Tính tay

| i | Các phần cuối `k` | Công thức | dp[i] |
|---|---|---|---|
| 0 | — | quy ước | 1 |
| 1 | 1 | dp[0] | 1 |
| 2 | 1 | dp[1] | 1 |
| 3 | 1, 3 | dp[2] + dp[0] | 2 |
| 4 | 1, 3 | dp[3] + dp[1] | 3 |
| 5 | 1, 3, 5 | dp[4] + dp[2] + dp[0] | 5 |
| 6 | 1, 3, 5 | dp[5] + dp[3] + dp[1] | 8 |

Đáp án `= 2 · dp[n]`. Ví dụ `n = 4`: `2 · 3 = 6` ✓

In [18]:
def odd_composition_table(n: int):
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        total = 0
        for k in range(1, i + 1, 2):      # độ dài phần cuối: 1, 3, 5, ...
            total += dp[i - k]
        dp[i] = total % MOD
    return dp

def count_good_strings_dp_quadratic(n: int) -> int:
    return 2 * odd_composition_table(n)[n] % MOD

print("dp[0..12] =", odd_composition_table(12))
for n, expected, _ in examples:
    assert count_good_strings_dp_quadratic(n) == expected
for n in range(1, 13):
    assert count_good_strings_dp_quadratic(n) == count_good_strings_bruteforce(n)
print("Cách 2 khớp ví dụ và brute force với n = 1..12")
print("n = 1000 ->", count_good_strings_dp_quadratic(1000))

dp[0..12] = [1, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144]
Cách 2 khớp ví dụ và brute force với n = 1..12
n = 1000 -> 35383207


## Độ phức tạp Cách 2

Vòng ngoài `i = 1..n`, vòng trong khoảng `i / 2` bước → tổng ≈ `n^2 / 4` phép cộng. **Time `O(n^2)`**, **Space `O(n)`**.

Với `n = 10^15` vẫn không thể. Nhưng hãy **nhìn kỹ dãy `dp`** vừa in: `1, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144`...

# 9. Bước đột phá 3 — Đó là dãy Fibonacci!

Bỏ `dp[0]` đi: `dp[1], dp[2], ... = 1, 1, 2, 3, 5, 8, 13, ...` = `F(1), F(2), F(3), ...`

## Chứng minh 1 (đại số)

Với `i ≥ 3`:

```text
dp[i]   = dp[i-1] + (dp[i-3] + dp[i-5] + ...)
dp[i-2] =            dp[i-3] + dp[i-5] + ...      (áp dụng công thức cho i-2 ≥ 1)
⇒ dp[i] = dp[i-1] + dp[i-2]
```

**Cẩn thận ở `i = 2`:** `dp[2] = dp[1] = 1`, nhưng `dp[1] + dp[0] = 2`. Công thức Fibonacci **không** đúng ở `i = 2` vì `dp[0] = 1` là *quy ước*, không phải một tổng. Vì vậy ta dùng `dp[1] = dp[2] = 1` rồi mới áp dụng Fibonacci từ `i = 3` — khớp đúng `F(1) = F(2) = 1`.

## Chứng minh 2 (song ánh — trực giác)

Lấy một composition lẻ của `i` (`i ≥ 3`) và nhìn **phần cuối**:

- Phần cuối `= 1` → **bỏ nó đi** → được một composition lẻ của `i - 1`.
- Phần cuối `≥ 3` → **trừ nó đi 2** (vẫn lẻ, vẫn ≥ 1) → được một composition lẻ của `i - 2`.

Phép biến đổi đảo ngược được (thêm `+1` ở cuối / cộng 2 vào phần cuối), hai nhóm rời nhau và phủ hết. Do đó `dp[i] = dp[i-1] + dp[i-2]`.

## Kết luận

> **Đáp án = 2 · F(n) mod (10^9 + 7)**, với `F(1) = F(2) = 1`.

Kiểm tra: `n = 2 → 2·1 = 2`, `n = 3 → 2·2 = 4`, `n = 4 → 2·3 = 6` ✓

In [19]:
dp = odd_composition_table(60)
F = fib_list(61)
assert all(dp[i] == (dp[i - 1] + dp[i - 2]) % MOD for i in range(3, 61))
assert all(dp[i] == F[i] % MOD for i in range(1, 61))
print("dp[i] = dp[i-1] + dp[i-2] đúng với i = 3..60, và dp[i] = F(i) với i = 1..60")
print()

# Minh họa song ánh với i = 5
for parts in odd_compositions(5):
    if parts[-1] == 1:
        image = parts[:-1]
        print(f"{str(parts):17} phần cuối = 1, bỏ đi    -> {image} (composition của 4)")
    else:
        image = parts[:-1] + [parts[-1] - 2]
        print(f"{str(parts):17} phần cuối >= 3, trừ 2 -> {image} (composition của 3)")

dp[i] = dp[i-1] + dp[i-2] đúng với i = 3..60, và dp[i] = F(i) với i = 1..60

[1, 1, 1, 1, 1]   phần cuối = 1, bỏ đi    -> [1, 1, 1, 1] (composition của 4)
[1, 1, 3]         phần cuối >= 3, trừ 2 -> [1, 1, 1] (composition của 3)
[1, 3, 1]         phần cuối = 1, bỏ đi    -> [1, 3] (composition của 4)
[3, 1, 1]         phần cuối = 1, bỏ đi    -> [3, 1] (composition của 4)
[5]               phần cuối >= 3, trừ 2 -> [3] (composition của 3)


# 10. Cách 3 — DP trạng thái (automaton), `O(n)` thời gian, `O(1)` bộ nhớ

Bây giờ ta nhìn bài toán theo một góc khác, **tổng quát hơn** và dẫn thẳng tới ma trận: **đọc chuỗi từ trái sang phải**, mỗi lần thêm một ký tự, và chỉ nhớ **run hiện tại đang dài lẻ hay chẵn**.

## Trạng thái

- `odd[i]` = số chuỗi độ dài `i` mà mọi run **đã kết thúc** đều lẻ, và run **cuối** (đang mở) có độ dài **lẻ**.
- `even[i]` = như trên, nhưng run cuối có độ dài **chẵn**.

## Chuyển trạng thái khi thêm ký tự thứ `i + 1`

Mỗi chuỗi có đúng 2 lựa chọn: thêm ký tự **giống** ký tự cuối, hoặc **khác**.

| Đang ở | Thêm ký tự **giống** | Thêm ký tự **khác** |
|---|---|---|
| `odd` (run cuối lẻ) | run dài thêm 1 → `even` | đóng run lẻ (hợp lệ), mở run mới dài 1 → `odd` |
| `even` (run cuối chẵn) | run dài thêm 1 → `odd` | đóng run **chẵn** ✗ **cấm** |

Đọc theo cột "đi tới":

```text
odd[i+1]  = odd[i] + even[i]     (odd + ký tự khác,  even + ký tự giống)
even[i+1] = odd[i]               (odd + ký tự giống)
```

**Khởi đầu:** `i = 1`: `"a"`, `"b"` đều là run dài 1 → `odd[1] = 2`, `even[1] = 0`.

**Đáp án:** `odd[n]` (chuỗi phải kết thúc bằng một run lẻ).

## Cú pháp gán đồng thời

```python
odd, even = (odd + even) % MOD, odd
```

Python tính **toàn bộ vế phải bằng giá trị cũ** rồi mới gán, nên `even` nhận `odd` **cũ** — không cần biến tạm.

> Nhận xét: thế `even[i] = odd[i-1]` vào dòng đầu được `odd[i+1] = odd[i] + odd[i-1]` — lại là Fibonacci (nhân 2). Hai góc nhìn khớp nhau.

In [20]:
def count_good_strings_dp_linear(n: int) -> int:
    odd, even = 2, 0              # i = 1: "a", "b" đều là run dài 1 (lẻ)
    for _ in range(n - 1):        # thêm lần lượt ký tự thứ 2, 3, ..., n
        odd, even = (odd + even) % MOD, odd
    return odd

for n, expected, _ in examples:
    assert count_good_strings_dp_linear(n) == expected
assert all(count_good_strings_dp_linear(n) == count_good_strings_dp_quadratic(n) for n in range(1, 201))
print("n = 1..12:", [count_good_strings_dp_linear(n) for n in range(1, 13)])
print("n = 10^5 ->", count_good_strings_dp_linear(10**5))

n = 1..12: [2, 2, 4, 6, 10, 16, 26, 42, 68, 110, 178, 288]
n = 10^5 -> 822870997


## Kiểm chứng ý nghĩa của `odd[i]`, `even[i]`

Đừng tin công thức chỉ vì nó ra đúng đáp án cuối. Cell dưới **đếm thật** bằng brute force: với mỗi chuỗi độ dài `i`, nếu mọi run (trừ run cuối) đều lẻ thì xếp vào nhóm `odd`/`even` theo độ dài run cuối, rồi so với DP.

In [21]:
def classify_prefix(s):
    """'odd'/'even' theo độ dài run cuối, nếu mọi run trước đó đều lẻ; ngược lại None."""
    lengths = run_lengths(s)
    if any(length % 2 == 0 for length in lengths[:-1]):
        return None
    return "odd" if lengths[-1] % 2 == 1 else "even"

odd, even = 2, 0
for i in range(1, 8):
    labels = [classify_prefix("".join(t)) for t in product("ab", repeat=i)]
    print(f"i = {i}: DP odd = {odd:2d}, even = {even:2d} | đếm thật odd = {labels.count('odd'):2d}, even = {labels.count('even'):2d}")
    assert (odd, even) == (labels.count("odd"), labels.count("even"))
    odd, even = odd + even, odd

i = 1: DP odd =  2, even =  0 | đếm thật odd =  2, even =  0
i = 2: DP odd =  2, even =  2 | đếm thật odd =  2, even =  2
i = 3: DP odd =  4, even =  2 | đếm thật odd =  4, even =  2
i = 4: DP odd =  6, even =  4 | đếm thật odd =  6, even =  4
i = 5: DP odd = 10, even =  6 | đếm thật odd = 10, even =  6
i = 6: DP odd = 16, even = 10 | đếm thật odd = 16, even = 10
i = 7: DP odd = 26, even = 16 | đếm thật odd = 26, even = 16


## Độ phức tạp Cách 3

- **Time `O(n)`**: một vòng lặp `n - 1` bước, mỗi bước vài phép tính.
- **Space `O(1)`**: chỉ hai biến `odd`, `even`.

Tốt hơn hẳn `O(n^2)`, nhưng `n = 10^15` vẫn quá lớn. Cell dưới đo thời gian với `n = 10^6` rồi nhân lên để ước lượng. Con số in ra phụ thuộc máy của bạn, nhưng luôn được tính bằng **năm** chứ không phải giây — trong khi LeetCode chỉ cho khoảng vài giây.

In [22]:
import time

start = time.perf_counter()
count_good_strings_dp_linear(10**6)
elapsed = time.perf_counter() - start
print(f"n = 10^6 mất khoảng {elapsed:.3f} giây")
print(f"Ước lượng cho n = 10^15: khoảng {elapsed * 10**9 / (3600 * 24 * 365):.1f} năm")

n = 10^6 mất khoảng 0.069 giây
Ước lượng cho n = 10^15: khoảng 2.2 năm


# 11. Cách 4 — Lũy thừa ma trận, `O(log n)`

## Viết bước cập nhật thành phép nhân ma trận

Bước cập nhật của Cách 3:

```text
odd'  = 1·odd + 1·even
even' = 1·odd + 0·even
```

chính là phép nhân ma trận với vector (Mục 4.8):

```text
[odd' ]   [[1, 1],   [odd ]
[even'] =  [1, 0]] · [even]          M = [[1, 1], [1, 0]]
```

Áp dụng `n - 1` lần, bắt đầu từ `[odd_1, even_1] = [2, 0]`:

```text
[odd_n, even_n] = M^(n-1) · [2, 0]   ⇒   odd_n = 2 · (M^(n-1))[0][0]
```

## Tính `M^(n-1)` thế nào cho nhanh?

Dùng **lũy thừa nhanh** (Mục 4.7) nhưng thay:

- số `1` → ma trận đơn vị `I`,
- "nhân số" → `mat_mul` (có `% MOD` ở mỗi phần tử).

Vì phép nhân ma trận có tính kết hợp nên lập luận "`a^13 = a^8 · a^4 · a^1`" vẫn đúng với ma trận.

Trường hợp `n = 1`: `M^0 = I`, `I[0][0] = 1` → đáp án `2` ✓

Số phép tính: khoảng 50 vòng, mỗi vòng tối đa 2 phép nhân ma trận, mỗi phép nhân ma trận 8 phép nhân số → **vài trăm phép nhân** thay vì `10^15`.

In [23]:
def mat_mul(A, B):
    """Nhân hai ma trận 2x2, lấy mod từng phần tử."""
    return [
        [(A[0][0] * B[0][0] + A[0][1] * B[1][0]) % MOD, (A[0][0] * B[0][1] + A[0][1] * B[1][1]) % MOD],
        [(A[1][0] * B[0][0] + A[1][1] * B[1][0]) % MOD, (A[1][0] * B[0][1] + A[1][1] * B[1][1]) % MOD],
    ]

def mat_pow(M, e):
    result = [[1, 0], [0, 1]]     # ma trận đơn vị (vai trò số 1)
    base = M
    while e > 0:
        if e & 1:
            result = mat_mul(result, base)
        base = mat_mul(base, base)
        e >>= 1
    return result

def count_good_strings_matrix(n: int) -> int:
    P = mat_pow([[1, 1], [1, 0]], n - 1)
    # [odd_n, even_n] = P · [2, 0]  ->  odd_n = P[0][0]*2 + P[0][1]*0
    return (P[0][0] * 2 + P[0][1] * 0) % MOD

for n, expected, _ in examples:
    assert count_good_strings_matrix(n) == expected
assert all(count_good_strings_matrix(n) == count_good_strings_dp_linear(n) for n in range(1, 500))
print("n = 1..12:", [count_good_strings_matrix(n) for n in range(1, 13)])
print("n = 10^15 ->", count_good_strings_matrix(10**15))

n = 1..12: [2, 2, 4, 6, 10, 16, 26, 42, 68, 110, 178, 288]
n = 10^15 -> 296650267


# 12. Cách 5 — Fast doubling Fibonacci, `O(log n)`

## Vì sao `M^k` chứa các số Fibonacci?

Khẳng định: `M^k = [[F(k+1), F(k)], [F(k), F(k-1)]]` với mọi `k ≥ 1`.

- `k = 1`: `M = [[1, 1], [1, 0]] = [[F(2), F(1)], [F(1), F(0)]]` ✓
- Giả sử đúng với `k`. Khi đó

```text
M^(k+1) = M^k · M = [[F(k+1) + F(k), F(k+1)],   = [[F(k+2), F(k+1)],
                     [F(k) + F(k-1), F(k)  ]]      [F(k+1), F(k)  ]]   ✓
```

Vậy `(M^(n-1))[0][0] = F(n)` và đáp án Cách 4 đúng là `2·F(n)` — khớp Bước đột phá 3.

## Công thức nhân đôi

Vì `M^(2k) = M^k · M^k`, so sánh phần tử ở góc trên-phải và trên-trái:

```text
F(2k)   = F(k+1)·F(k) + F(k)·F(k-1) = F(k) · (F(k+1) + F(k-1))
        = F(k) · (2·F(k+1) - F(k))              (vì F(k-1) = F(k+1) - F(k))

F(2k+1) = F(k+1)^2 + F(k)^2
```

Nghĩa là: **biết cặp `(F(k), F(k+1))` thì tính ngay được `(F(2k), F(2k+1))`** chỉ với 3 phép nhân. Muốn bước sang `2k + 1` thì thêm: `F(2k+2) = F(2k) + F(2k+1)`.

Kiểm tra số trước khi tin:

In [24]:
F = fib_list(80)
for k in range(1, 31):
    assert F[2 * k] == F[k] * (2 * F[k + 1] - F[k])
    assert F[2 * k + 1] == F[k + 1] ** 2 + F[k] ** 2
print("Hai công thức đúng với k = 1..30")

k = 6
f2k = F[k] * (2 * F[k + 1] - F[k])
f2k1 = F[k + 1] ** 2 + F[k] ** 2
print(f"k = {k}: F(k) = {F[k]}, F(k+1) = {F[k+1]} -> F(2k) = {f2k} (thật: {F[2*k]}), F(2k+1) = {f2k1} (thật: {F[2*k+1]})")

Hai công thức đúng với k = 1..30
k = 6: F(k) = 8, F(k+1) = 13 -> F(2k) = 144 (thật: 144), F(2k+1) = 233 (thật: 233)


## Thuật toán đệ quy

`fib_pair(k)` trả về cặp `(F(k), F(k+1))` theo modulo:

1. Nếu `k = 0` → `(0, 1)`.
2. Gọi `m = k // 2` (tức `k >> 1`), lấy `(a, b) = fib_pair(m) = (F(m), F(m+1))`.
3. Tính `c = F(2m) = a·(2b − a)`, `d = F(2m+1) = a² + b²` (đều `% MOD`).
4. Nếu `k` chẵn (`k = 2m`) → trả `(c, d)`; nếu `k` lẻ (`k = 2m + 1`) → trả `(d, c + d)`.

Mỗi lần gọi, `k` giảm một nửa → độ sâu đệ quy ≈ số bit của `n` ≈ 50 (rất xa giới hạn đệ quy của Python).

Lưu ý `(2·b − a)` có thể âm khi đã lấy mod (vì `a`, `b` là phần dư). Python `%` xử lý đúng; C++/Java phải viết `(2*b - a + MOD) % MOD`.

So với Cách 4: cùng `O(log n)` nhưng mỗi bước chỉ ~3 phép nhân thay vì tới 16 → nhanh hơn vài lần, code ngắn hơn.

In [25]:
def fib_pair(k: int):
    """Trả về (F(k), F(k+1)) theo modulo MOD."""
    if k == 0:
        return (0, 1)
    a, b = fib_pair(k >> 1)                 # a = F(m), b = F(m+1), m = k // 2
    c = a * ((2 * b - a) % MOD) % MOD       # F(2m)
    d = (a * a + b * b) % MOD               # F(2m+1)
    if k & 1:
        return (d, (c + d) % MOD)           # k = 2m+1 -> (F(2m+1), F(2m+2))
    return (c, d)                           # k = 2m   -> (F(2m),   F(2m+1))

def count_good_strings_doubling(n: int) -> int:
    return 2 * fib_pair(n)[0] % MOD

for n, expected, _ in examples:
    assert count_good_strings_doubling(n) == expected
assert all(count_good_strings_doubling(n) == count_good_strings_matrix(n) for n in range(1, 2000))
print("n = 1..12:", [count_good_strings_doubling(n) for n in range(1, 13)])
print("n = 10^15 ->", count_good_strings_doubling(10**15))
print("Khớp Cách 4?", count_good_strings_doubling(10**15) == count_good_strings_matrix(10**15))

n = 1..12: [2, 2, 4, 6, 10, 16, 26, 42, 68, 110, 178, 288]
n = 10^15 -> 296650267
Khớp Cách 4? True


# 13. Mô phỏng từng bước (Trace)

## 13.1 DP trạng thái (Cách 3) với `n = 6`

Ý nghĩa các cột: `odd` / `even` = số chuỗi độ dài `i` (mọi run đã đóng đều lẻ) có run cuối lẻ / chẵn.

| i | odd[i] | even[i] | Cách tính | Ví dụ |
|---|---:|---:|---|---|
| 1 | 2 | 0 | khởi đầu | odd: `a`, `b` |
| 2 | 2 | 2 | odd = 2 + 0, even = odd cũ = 2 | odd: `ab`, `ba`; even: `aa`, `bb` |
| 3 | 4 | 2 | 2 + 2, 2 | odd: `aaa`, `aba`, `bab`, `bbb` |
| 4 | 6 | 4 | 4 + 2, 4 | even: `aaaa`, `abaa`, `babb`, `bbbb` |
| 5 | 10 | 6 | 6 + 4, 6 | |
| 6 | 16 | 10 | 10 + 6, 10 | |

Đáp án `n = 6` là `odd[6] = 16` (đúng bằng `2·F(6) = 2·8`).

In [26]:
def trace_dp_states(n: int) -> int:
    odd, even = 2, 0
    print(f"i = 1: odd = {odd:3d}, even = {even:3d}   (khởi đầu: 'a', 'b')")
    for i in range(2, n + 1):
        new_odd, new_even = (odd + even) % MOD, odd
        print(f"i = {i}: odd = {odd} + {even} = {new_odd:3d}, even = odd cũ = {new_even:3d}")
        odd, even = new_odd, new_even
    print("Đáp án = odd[n] =", odd)
    return odd

trace_dp_states(6)

i = 1: odd =   2, even =   0   (khởi đầu: 'a', 'b')
i = 2: odd = 2 + 0 =   2, even = odd cũ =   2
i = 3: odd = 2 + 2 =   4, even = odd cũ =   2
i = 4: odd = 4 + 2 =   6, even = odd cũ =   4
i = 5: odd = 6 + 4 =  10, even = odd cũ =   6
i = 6: odd = 10 + 6 =  16, even = odd cũ =  10
Đáp án = odd[n] = 16


16

## 13.2 Lũy thừa ma trận (Cách 4) với `n = 13`

Cần `M^12`, và `12 = 1100₂`. Duyệt bit từ phải sang trái; `base` lần lượt là `M^1, M^2, M^4, M^8`.

| Vòng | e (nhị phân) | bit cuối | Hành động | result sau vòng |
|---|---|---|---|---|
| 1 | `1100` | 0 | bỏ qua (base = M^1) | `I = [[1, 0], [0, 1]]` |
| 2 | `110` | 0 | bỏ qua (base = M^2) | `I` |
| 3 | `11` | 1 | nhân base = M^4 = `[[5, 3], [3, 2]]` | `[[5, 3], [3, 2]]` |
| 4 | `1` | 1 | nhân base = M^8 = `[[34, 21], [21, 13]]` | `M^12 = [[233, 144], [144, 89]]` |

Đáp án `= 2 · 233 = 466` (= `2·F(13)`). Chỉ 4 vòng thay vì 12 phép nhân.

In [27]:
def trace_matrix_power(n: int) -> int:
    e = n - 1
    result = [[1, 0], [0, 1]]
    base = [[1, 1], [1, 0]]
    power = 1                  # base hiện tại = M^power
    step = 1
    print(f"n = {n}: cần M^{e}, e = {bin(e)[2:]} (nhị phân)")
    while e > 0:
        bit = e & 1
        if bit:
            result = mat_mul(result, base)
            action = f"nhân base = M^{power}"
        else:
            action = f"bỏ qua (base = M^{power})"
        print(f"Vòng {step}: e = {bin(e)[2:]:>4}, bit = {bit}, {action:22} -> result = {result}")
        base = mat_mul(base, base)
        power *= 2
        e >>= 1
        step += 1
    answer = 2 * result[0][0] % MOD
    print("Đáp án = 2 * result[0][0] =", answer)
    return answer

trace_matrix_power(13)

n = 13: cần M^12, e = 1100 (nhị phân)
Vòng 1: e = 1100, bit = 0, bỏ qua (base = M^1)    -> result = [[1, 0], [0, 1]]
Vòng 2: e =  110, bit = 0, bỏ qua (base = M^2)    -> result = [[1, 0], [0, 1]]
Vòng 3: e =   11, bit = 1, nhân base = M^4        -> result = [[5, 3], [3, 2]]
Vòng 4: e =    1, bit = 1, nhân base = M^8        -> result = [[233, 144], [144, 89]]
Đáp án = 2 * result[0][0] = 466


466

## 13.3 Fast doubling (Cách 5) với `n = 13`

Chuỗi lời gọi: `13 → 6 → 3 → 1 → 0` (mỗi lần `k // 2`). Kết quả được tính **khi quay về**:

| k | m = k // 2 | (F(m), F(m+1)) | c = F(2m) | d = F(2m+1) | k chẵn/lẻ | Trả về (F(k), F(k+1)) |
|---|---|---|---|---|---|---|
| 0 | — | — | — | — | — | `(0, 1)` |
| 1 | 0 | `(0, 1)` | 0 | 1 | lẻ | `(1, 1)` |
| 3 | 1 | `(1, 1)` | 1 | 2 | lẻ | `(2, 3)` |
| 6 | 3 | `(2, 3)` | 8 | 13 | chẵn | `(8, 13)` |
| 13 | 6 | `(8, 13)` | 144 | 233 | lẻ | `(233, 377)` |

`F(13) = 233` → đáp án `466`, khớp với trace ma trận.

In [28]:
def trace_fast_doubling(k: int, depth: int = 0):
    indent = "    " * depth
    if k == 0:
        print(f"{indent}k = 0 -> trả về (F(0), F(1)) = (0, 1)")
        return (0, 1)
    print(f"{indent}k = {k}: gọi đệ quy với m = {k >> 1}")
    a, b = trace_fast_doubling(k >> 1, depth + 1)
    c = a * ((2 * b - a) % MOD) % MOD
    d = (a * a + b * b) % MOD
    res = (d, (c + d) % MOD) if k & 1 else (c, d)
    kind = "lẻ" if k & 1 else "chẵn"
    m = k >> 1
    print(f"{indent}k = {k}: (F({m}), F({m+1})) = ({a}, {b}) -> c = F({2*m}) = {c}, d = F({2*m+1}) = {d}; k {kind} -> trả về {res}")
    return res

F13, _ = trace_fast_doubling(13)
print("Đáp án n = 13:", 2 * F13 % MOD)

k = 13: gọi đệ quy với m = 6
    k = 6: gọi đệ quy với m = 3
        k = 3: gọi đệ quy với m = 1
            k = 1: gọi đệ quy với m = 0
                k = 0 -> trả về (F(0), F(1)) = (0, 1)
            k = 1: (F(0), F(1)) = (0, 1) -> c = F(0) = 0, d = F(1) = 1; k lẻ -> trả về (1, 1)
        k = 3: (F(1), F(2)) = (1, 1) -> c = F(2) = 1, d = F(3) = 2; k lẻ -> trả về (2, 3)
    k = 6: (F(3), F(4)) = (2, 3) -> c = F(6) = 8, d = F(7) = 13; k chẵn -> trả về (8, 13)
k = 13: (F(6), F(7)) = (8, 13) -> c = F(12) = 144, d = F(13) = 233; k lẻ -> trả về (233, 377)
Đáp án n = 13: 466


# 14. Phiên bản nộp LeetCode

Ta dùng **fast doubling dạng vòng lặp** để không cần đệ quy: duyệt các bit của `n` **từ trái sang phải** (bit cao trước).

- Bắt đầu `k = 0`, `(a, b) = (F(0), F(1)) = (0, 1)`.
- Mỗi bit: nhân đôi `k → 2k` bằng công thức doubling; nếu bit là `1` thì tiến thêm `2k → 2k + 1`.
- **Bất biến:** sau khi đọc một tiền tố nhị phân của `n` (ứng với số `k`), luôn có `(a, b) = (F(k), F(k+1))`. Đọc hết các bit thì `k = n`.

Ví dụ `n = 13 = 1101₂`: `k` đi qua `0 → 1 → 3 → 6 → 13` — đúng các giá trị trong bảng 13.3, chỉ là theo chiều ngược lại (từ dưới lên).

Chỉ ~50 vòng lặp với `n = 10^15`, mỗi vòng vài phép nhân số nhỏ hơn `MOD^2` → chạy trong vài micro giây, không có rủi ro TLE dù dùng Python thuần.

> Tên phương thức `countGoodStrings` được suy ra từ slug `count-good-strings`; nếu editor LeetCode dùng tên khác, chỉ cần đổi tên hàm, phần thân giữ nguyên.

In [29]:
class Solution:
    def countGoodStrings(self, n: int) -> int:
        MOD = 10**9 + 7
        a, b = 0, 1                              # (F(k), F(k+1)) với k = 0
        for bit in bin(n)[2:]:                   # duyệt bit của n từ trái sang phải
            c = a * ((2 * b - a) % MOD) % MOD    # F(2k)
            d = (a * a + b * b) % MOD            # F(2k+1)
            if bit == "1":
                a, b = d, (c + d) % MOD          # k -> 2k + 1
            else:
                a, b = c, d                      # k -> 2k
        return 2 * a % MOD

In [30]:
sol = Solution()

tests = [
    (4, 6), (3, 4), (2, 2),          # ba ví dụ trong đề
    (1, 2),                          # n nhỏ nhất: "a", "b"
    (5, 10), (6, 16), (10, 110),     # giá trị nhỏ đã kiểm bằng brute force
]
for n, expected in tests:
    result = sol.countGoodStrings(n)
    print(f"n = {n:>2} -> {result:>4} | expected = {expected:>4} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

big = 10**15
assert sol.countGoodStrings(big) == count_good_strings_matrix(big) == count_good_strings_doubling(big)
print("n = 10^15 ->", sol.countGoodStrings(big))
assert all(sol.countGoodStrings(n) == count_good_strings_dp_linear(n) for n in range(1, 300))
print("Tất cả test PASS!")

n =  4 ->    6 | expected =    6 | PASS
n =  3 ->    4 | expected =    4 | PASS
n =  2 ->    2 | expected =    2 | PASS
n =  1 ->    2 | expected =    2 | PASS
n =  5 ->   10 | expected =   10 | PASS
n =  6 ->   16 | expected =   16 | PASS
n = 10 ->  110 | expected =  110 | PASS
n = 10^15 -> 296650267
Tất cả test PASS!


# 15. Kiểm tra chéo (cross-check) mọi cách giải

Ý tưởng: dùng các cách **chậm nhưng chắc đúng** làm chuẩn cho các cách nhanh.

1. `n` nhỏ (≤ 12): so **mọi** cách với **brute force theo định nghĩa** (Cách 1).
2. `n` vừa (≤ 3000): so Cách 4, Cách 5, `Solution` với DP `O(n)` (Cách 3).
3. `n` khổng lồ (≤ `10^15`): so Cách 5 và `Solution` với Cách 4 (hai thuật toán khác nhau mà cùng ra một số là bằng chứng mạnh).

`random.seed(...)` cố định để mỗi lần chạy notebook đều cho cùng bộ test.

In [31]:
import random
random.seed(4073)

approaches = {
    "dp_quadratic": count_good_strings_dp_quadratic,
    "dp_linear": count_good_strings_dp_linear,
    "matrix": count_good_strings_matrix,
    "doubling": count_good_strings_doubling,
    "Solution": sol.countGoodStrings,
}

small_cases = 0
for _ in range(60):
    n = random.randint(1, 12)
    ref = count_good_strings_bruteforce(n)
    for name, f in approaches.items():
        assert f(n) == ref, (name, n)
    small_cases += 1

medium_cases = 0
for _ in range(200):
    n = random.randint(1, 3000)
    ref = count_good_strings_dp_linear(n)
    assert count_good_strings_matrix(n) == ref
    assert count_good_strings_doubling(n) == ref
    assert sol.countGoodStrings(n) == ref
    medium_cases += 1

big_cases = 0
for _ in range(300):
    n = random.randint(1, 10**15)
    ref = count_good_strings_matrix(n)
    assert count_good_strings_doubling(n) == ref and sol.countGoodStrings(n) == ref
    big_cases += 1

print("n nhỏ  (so với brute force):", small_cases, "ca PASS")
print("n vừa  (so với DP O(n))    :", medium_cases, "ca PASS")
print("n tới 10^15 (matrix/doubling):", big_cases, "ca PASS")

n nhỏ  (so với brute force): 60 ca PASS
n vừa  (so với DP O(n))    : 200 ca PASS
n tới 10^15 (matrix/doubling): 300 ca PASS


# 16. Vì sao thuật toán đúng?

Chuỗi lập luận gồm 4 mắt xích, mỗi mắt xích đã được chứng minh (và kiểm bằng code) ở trên:

1. **Bổ đề run** (Bước đột phá 1): `s` tốt ⇔ mọi run của `s` có độ dài lẻ.
2. **Song ánh** (Bước đột phá 2): chuỗi tốt độ dài `n` ↔ cặp (ký tự đầu ∈ {`a`, `b`}, composition lẻ của `n`). Vì các run luân phiên ký tự, biết ký tự đầu và độ dài các run là dựng lại được đúng một chuỗi, và ngược lại. ⇒ đáp án `= 2 · C(n)`.
3. **Fibonacci** (Bước đột phá 3): `C(n) = F(n)` với `F(1) = F(2) = 1`.
4. **Tính đúng `F(n) mod p`**: modulo bảo toàn phép cộng và nhân, nên lấy mod sau mỗi bước không làm đổi kết quả cuối.

## Bất biến (invariant) của từng vòng lặp

- **Cách 3:** sau khi xử lý `i` ký tự, `(odd, even) = (odd[i], even[i]) mod p`.
- **Cách 4:** trước mỗi vòng, `result · base^e = M^(n-1)`. Bit cuối của `e` bằng 1 thì chuyển một thừa số `base` sang `result`; sau đó `base` bình phương và `e` giảm một nửa — đẳng thức vẫn giữ. Khi `e = 0` thì `result = M^(n-1)`.
- **Cách 5 / `Solution`:** sau khi đọc một tiền tố nhị phân của `n` ứng với số `k`, `(a, b) = (F(k), F(k+1))`. Đọc thêm bit `0` thì `k → 2k`, bit `1` thì `k → 2k + 1`; công thức doubling giữ đúng bất biến.

# 17. Độ phức tạp

Gọi `n` là độ dài chuỗi (input). Với các cách `O(log n)`, mỗi phép tính là trên số nhỏ hơn `MOD^2 ≈ 10^18` nên xem là `O(1)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1 | Brute force theo định nghĩa | `O(2^n · n^2)` | `O(2^n · n)` (cache) | chỉ `n ≤ ~15`; dùng làm chuẩn |
| 2 | DP composition theo phần cuối | `O(n^2)` | `O(n)` | |
| 3 | DP trạng thái chẵn/lẻ | `O(n)` | `O(1)` | đơn giản, tổng quát |
| 4 | Lũy thừa ma trận 2×2 | `O(log n)` | `O(1)` | ~50 vòng, ≤ 2 phép nhân ma trận mỗi vòng |
| 5 | Fast doubling | `O(log n)` | `O(log n)` (đệ quy) / `O(1)` (bản lặp trong `Solution`) | ~50 vòng, ~3 phép nhân mỗi vòng |

**Kết luận:** chỉ Cách 4 và Cách 5 đáp ứng `n = 10^15`. Cả hai đều chạy tức thì kể cả bằng Python thuần.

# 18. Những lỗi thường gặp

## Lỗi 1 — Quên nhân 2

Chỉ đếm chuỗi bắt đầu bằng `'a'` (tức `F(n)`), quên chuỗi bắt đầu bằng `'b'`.

```python
return fib(n)            # sai: n = 4 ra 3
return 2 * fib(n) % MOD  # đúng
```

## Lỗi 2 — Lệch chỉ số Fibonacci

Có nhiều quy ước (`F(0) = 0` hay `F(1) = 0`...). Viết nhầm `2 · F(n+1)` hoặc dùng `M^n` thay cho `M^(n-1)` → lệch một bậc. **Luôn thử lại với `n = 1, 2, 3, 4`** (đáp án `2, 2, 4, 6`).

## Lỗi 3 — Hiểu sai đề

- Nghĩ "chuỗi tốt = chuỗi xen kẽ" (không có hai ký tự giống nhau đứng cạnh) → bỏ sót `"aaab"`, `"abbb"`...
- Nghĩ `"aa" = "a" + "a"` là tốt → quên điều kiện "cuối `s1` khác đầu `s2`".

Cell dưới chạy thử các phiên bản sai để thấy chúng lệch ở đâu.

In [32]:
def wrong_forget_two(n):            # Lỗi 1: quên nhân 2
    return fib_pair(n)[0]

def wrong_index(n):                 # Lỗi 2: lệch chỉ số, dùng F(n+1)
    return 2 * fib_pair(n + 1)[0] % MOD

def wrong_alternating(n):           # Lỗi 3: chỉ đếm chuỗi xen kẽ
    return sum(1 for t in product("ab", repeat=n) if all(t[i] != t[i + 1] for i in range(n - 1)))

for n in [2, 3, 4]:
    print(f"n = {n}: đúng = {count_good_strings_doubling(n)}, quên x2 = {wrong_forget_two(n)}, "
          f"lệch chỉ số = {wrong_index(n)}, chỉ xen kẽ = {wrong_alternating(n)}")

n = 2: đúng = 2, quên x2 = 1, lệch chỉ số = 4, chỉ xen kẽ = 2
n = 3: đúng = 4, quên x2 = 2, lệch chỉ số = 6, chỉ xen kẽ = 2
n = 4: đúng = 6, quên x2 = 3, lệch chỉ số = 10, chỉ xen kẽ = 2


## Lỗi 4 — Dùng công thức Binet với số thực (float)

Về toán, `F(n) = round(φ^n / √5)` với `φ = (1 + √5) / 2`. Nhưng `float` chỉ giữ khoảng 15–16 chữ số có nghĩa → **sai từ khoảng `n = 71`**, còn với `n = 10^15` thì `φ^n` tràn số (`OverflowError`). Không bao giờ dùng float cho bài đếm có modulo.

## Lỗi 5 — Đệ quy sai kiểu

- Đệ quy Fibonacci không memo → số lần gọi tăng theo cấp số mũ (21891 lần gọi chỉ để tính `F(20)`, Mục 4.3).
- Đệ quy **tuyến tính** (`F(k)` gọi `F(k-1)`) → độ sâu `k` tầng → `RecursionError` khi `k` vượt giới hạn đệ quy của Python. Fast doubling chỉ sâu ~50 tầng nên an toàn.

## Lỗi 6 — Quên `% MOD` trong nhân ma trận

Python không tràn số nên kết quả *vẫn đúng* nếu lấy mod ở cuối — nhưng các số phình to khủng khiếp (`F(10^5)` đã có khoảng 20899 chữ số) và với `n = 10^15` thì không đủ bộ nhớ. Trong C++/Java thì **tràn số** ngay. Ngoài ra Python ≥ 3.11 còn từ chối `str()` một số nguyên quá 4300 chữ số (để tránh tấn công từ chối dịch vụ).

## Lỗi 7 — Phần dư âm ở ngôn ngữ khác

Trong fast doubling có `(2·b − a)`, có thể âm. Python `%` luôn cho kết quả không âm, nhưng C++/Java thì không: phải viết `(2*b - a + MOD) % MOD` và dùng kiểu `long long` / `long`.

In [33]:
import math, sys

# Lỗi 4: Binet bằng float
phi = (1 + 5 ** 0.5) / 2
F = fib_list(100)
first_bad = next(k for k in range(100) if round(phi ** k / 5 ** 0.5) != F[k])
print("Binet (float) sai lần đầu tại n =", first_bad)

# Lỗi 5: đệ quy tuyến tính -> RecursionError
def fib_pair_slow(k):
    """(F(k), F(k+1)) từ (F(k-1), F(k)) -> sâu k tầng."""
    if k == 0:
        return (0, 1)
    a, b = fib_pair_slow(k - 1)
    return (b, (a + b) % MOD)

print("fib_pair_slow(500)[0] =", fib_pair_slow(500)[0], "| fib_pair(500)[0] =", fib_pair(500)[0])
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())
try:
    fib_pair_slow(10**5)
except RecursionError as err:
    print("fib_pair_slow(10^5) -> RecursionError:", str(err)[:45])

# Lỗi 7: phần dư âm
a, b = 20, 5
print("Python: (2*b - a) % 7 =", (2 * b - a) % 7, "| kiểu C/Java (math.fmod):", math.fmod(2 * b - a, 7))

Binet (float) sai lần đầu tại n = 71
fib_pair_slow(500)[0] = 550656477 | fib_pair(500)[0] = 550656477
Giới hạn đệ quy hiện tại: 1000
fib_pair_slow(10^5) -> RecursionError: maximum recursion depth exceeded
Python: (2*b - a) % 7 = 4 | kiểu C/Java (math.fmod): -3.0


In [34]:
# Lỗi 6: lũy thừa ma trận KHÔNG lấy mod
def mat_pow_no_mod(M, e):
    result = [[1, 0], [0, 1]]
    while e > 0:
        if e & 1:
            result = mat_mul_plain(result, M)
        M = mat_mul_plain(M, M)
        e >>= 1
    return result

huge = mat_pow_no_mod([[1, 1], [1, 0]], 10**5 - 1)[0][0]     # = F(10^5), không lấy mod
print("F(10^5) có", huge.bit_length(), "bit, khoảng", int(huge.bit_length() * math.log10(2)) + 1, "chữ số")
print("Lấy mod ở cuối có ra đúng không?", huge % MOD == fib_pair(10**5)[0])
try:
    text = str(huge)
    print("str() thành công (Python cũ hơn 3.11), độ dài", len(text))
except ValueError as err:
    print("str() báo lỗi:", str(err)[:60], "...")

F(10^5) có 69424 bit, khoảng 20899 chữ số
Lấy mod ở cuối có ra đúng không? True
str() báo lỗi: Exceeds the limit (4300 digits) for integer string conversio ...


## Edge cases cần nhớ

| Input | Đáp án | Vì sao dễ sai |
|---|---|---|
| `n = 1` | 2 | `M^0 = I`; vòng lặp Cách 3 chạy 0 lần; `bin(1) = '0b1'` chỉ một bit |
| `n = 2` | 2 | chỗ công thức Fibonacci "bắt đầu" (`dp[2] = 1`, không phải `dp[1] + dp[0]`) |
| `n = 3` | 4 | ví dụ đề |
| `n = 4` | 6 | ví dụ đề, bắt được lỗi quên nhân 2 (ra 3) và lệch chỉ số (ra 10) |
| `n = 10^15` | 296650267 | chỉ chạy được bằng `O(log n)`, phải có `% MOD` |

Số `296650267` là kết quả mà cả Cách 4 và Cách 5 (hai thuật toán khác nhau) cùng in ra ở trên.

# 19. Dấu hiệu nhận dạng và bài liên quan

## Dấu hiệu 1 — `n` khổng lồ (`10^12 .. 10^18`) + "modulo `10^9 + 7`"

Không thể duyệt tới `n` ⇒ đi tìm **công thức truy hồi tuyến tính hệ số hằng** rồi dùng **lũy thừa ma trận** hoặc **fast doubling**.

## Dấu hiệu 2 — Định nghĩa đệ quy kiểu "ghép hai phần tốt"

Viết brute force bám sát định nghĩa, **in kết quả với `n` nhỏ**, rồi tìm quy luật: dãy quen thuộc (Fibonacci, Catalan, `2^n`, ...) hoặc đặc trưng đơn giản (ở đây: mọi run lẻ). Mẹo: tra dãy `2, 2, 4, 6, 10, 16, 26, ...` trên OEIS.

## Dấu hiệu 3 — Điều kiện về "khối ký tự giống nhau liên tiếp" (run)

Nghĩ tới **composition** (độ dài các run) hoặc **DP trạng thái** với trạng thái = độ dài / tính chẵn lẻ của run hiện tại.

## Dấu hiệu 4 — DP mà mỗi bước chỉ phụ thuộc `k` giá trị trước, hệ số cố định

Đóng gói `k` giá trị thành vector, bước cập nhật thành ma trận `k × k` ⇒ `O(k^3 log n)`.

## Bài liên quan trên LeetCode

| Bài | Liên hệ |
|---|---|
| 70. Climbing Stairs | composition với phần ∈ {1, 2} → Fibonacci |
| 509. Fibonacci Number | Fibonacci cơ bản |
| 1137. N-th Tribonacci Number | truy hồi 3 số hạng → ma trận 3×3 |
| 50. Pow(x, n) | lũy thừa nhanh với số |
| 790. Domino and Tromino Tiling | truy hồi tuyến tính từ bài đếm cách lát |
| 935. Knight Dialer | DP trạng thái, có thể tăng tốc bằng ma trận |
| 1220. Count Vowels Permutation | DP trạng thái trên chuỗi, ma trận 5×5 |
| 552. Student Attendance Record II | DP trạng thái theo "chuỗi ký tự liên tiếp" |

## Biến thể: bảng chữ cái có `k` ký tự

Nếu chuỗi được dùng `k` ký tự (vẫn yêu cầu mọi run lẻ) thì DP trạng thái chỉ đổi một chỗ: khi **đổi ký tự** có `k - 1` lựa chọn.

```text
odd[i+1]  = even[i] + (k - 1) · odd[i]
even[i+1] = odd[i]
odd[1] = k, even[1] = 0
```

Ma trận tương ứng: `[[k - 1, 1], [1, 0]]` — cùng khung lũy thừa ma trận như Cách 4. Đây là lý do DP trạng thái "tổng quát hơn" công thức Fibonacci: đổi đề một chút vẫn dùng được.

In [35]:
def count_odd_runs_k_letters(n: int, k: int) -> int:
    odd, even = k % MOD, 0
    for _ in range(n - 1):
        odd, even = (even + (k - 1) * odd) % MOD, odd
    return odd

def brute_odd_runs_k_letters(n: int, k: int) -> int:
    letters = "abcdefghijklmnopqrstuvwxyz"[:k]
    return sum(1 for t in product(letters, repeat=n) if all(L % 2 == 1 for L in run_lengths(t)))

for k in [2, 3]:
    dp_vals = [count_odd_runs_k_letters(n, k) for n in range(1, 8)]
    bf_vals = [brute_odd_runs_k_letters(n, k) for n in range(1, 8)]
    print(f"k = {k}: DP = {dp_vals}")
    assert dp_vals == bf_vals
print("Biến thể k ký tự khớp brute force với n = 1..7")

k = 2: DP = [2, 2, 4, 6, 10, 16, 26]
k = 3: DP = [3, 6, 15, 36, 87, 210, 507]
Biến thể k ký tự khớp brute force với n = 1..7


# 20. Bài tập vận dụng

Làm theo thứ tự, **đừng xem đáp án** trước khi tự thử. Gợi ý 3 mức nằm sau bài 6; đáp án ở cuối notebook.

## Bài 1 — Đoán output (không chạy code)

Chuỗi nào dưới đây là **tốt**?

```text
"aaabbb", "aabaa", "ababa", "abbba", "abbbba", "bbbbb"
```

Và đoán `countGoodStrings(7)`, `countGoodStrings(8)`.

In [36]:
bai1_strings = ["aaabbb", "aabaa", "ababa", "abbba", "abbbba", "bbbbb"]

# Ghi dự đoán của bạn trước, ví dụ: my_guess = [True, False, ...]
# Sau đó mới bỏ comment để kiểm tra:
# print([is_good_runs(s) for s in bai1_strings])
# print(sol.countGoodStrings(7), sol.countGoodStrings(8))

## Bài 2 — Tính bằng tay

a) Lập tiếp bảng `odd[i]`, `even[i]` của Mục 13.1 tới `i = 8`. Đáp án `n = 8` là bao nhiêu?

b) Tính `M^5` với `M = [[1, 1], [1, 0]]` bằng lũy thừa nhanh: `5 = 101₂` nên cần nhân những `M^(2^j)` nào?

c) Biết `F(5) = 5`, `F(6) = 8`. Dùng công thức doubling tính `F(10)` và `F(11)`.

## Bài 3 — Điền TODO: DP trạng thái

Hoàn thiện hàm dưới (Cách 3) mà không nhìn lại code phía trên.

In [37]:
def exercise_3_dp_linear(n: int) -> int:
    MOD = 10**9 + 7
    # TODO 1: khởi tạo odd, even cho i = 1
    # TODO 2: lặp n - 1 lần, cập nhật ĐỒNG THỜI:
    #         odd mới = odd + even, even mới = odd cũ   (nhớ % MOD)
    # TODO 3: trả về odd
    pass

# Làm xong rồi mới bỏ comment để test:
# assert exercise_3_dp_linear(4) == 6
# assert exercise_3_dp_linear(3) == 4
# assert exercise_3_dp_linear(1) == 2

## Bài 4 — Tự viết lại: lũy thừa ma trận

Viết từ đầu `mat_mul` (có `% MOD`), `mat_pow` (lũy thừa nhanh) và hàm đếm chuỗi tốt bằng ma trận. Phải chạy được `n = 10^15`.

In [38]:
def exercise_4_matrix(n: int) -> int:
    # TODO: tự viết mat_mul (có % MOD) và mat_pow (lũy thừa nhanh),
    #       rồi trả về 2 * (M^(n-1))[0][0] % MOD với M = [[1, 1], [1, 0]]
    raise NotImplementedError("Hãy tự viết trước khi xem đáp án")

# assert exercise_4_matrix(4) == 6
# assert exercise_4_matrix(1) == 2
# assert exercise_4_matrix(10**15) == Solution().countGoodStrings(10**15)

## Bài 5 — Biến thể: mọi run có độ dài **chẵn**

Đếm số chuỗi `'a'/'b'` độ dài `n` mà **mọi run có độ dài chẵn**, modulo `10^9 + 7`, với `1 <= n <= 10^15`.

Ví dụ: `n = 4` → 4 chuỗi (`aaaa`, `aabb`, `bbaa`, `bbbb`); `n = 3` → 0.

In [39]:
def exercise_5_even_runs(n: int) -> int:
    # TODO: suy nghĩ composition của n thành các phần CHẴN
    pass

# assert exercise_5_even_runs(3) == 0
# assert exercise_5_even_runs(4) == 4
# assert exercise_5_even_runs(2) == 2

## Bài 6 — Mini challenge: mỗi run dài 1 hoặc 3

Đếm số chuỗi `'a'/'b'` độ dài `n` mà **mỗi run có độ dài 1 hoặc 3**, modulo `10^9 + 7`, với `1 <= n <= 10^15`.

Giá trị nhỏ để tự kiểm tra: `n = 1..8` → `2, 2, 4, 6, 8, 12, 18, 26`.

Gợi ý lớn: đây **không còn** là Fibonacci; cần một ma trận 3×3.

In [40]:
def exercise_6_runs_1_or_3(n: int) -> int:
    # TODO: tìm truy hồi cho số composition của n với phần thuộc {1, 3},
    #       rồi dựng ma trận 3x3 và lũy thừa nhanh
    raise NotImplementedError

# assert [exercise_6_runs_1_or_3(n) for n in range(1, 9)] == [2, 2, 4, 6, 8, 12, 18, 26]

## Gợi ý (3 mức: hướng tư duy → cấu trúc/cú pháp → gần-code)

**Bài 1**
1. Đừng thử từng cách cắt — dùng Bước đột phá 1.
2. Viết độ dài các run của từng chuỗi.
3. `"aabaa"` → `[2, 1, 2]` có số chẵn → không tốt. Với `n = 7, 8`: `2·F(7)`, `2·F(8)`.

**Bài 2**
1. a) Dùng `odd' = odd + even`, `even' = odd`. b) Mỗi bit 1 ứng với một lần nhân vào `result`.
2. b) `5 = 4 + 1` → `result = M^1 · M^4`. c) Áp dụng `F(2k)` và `F(2k+1)` với `k = 5`.
3. c) `F(10) = F(5)·(2·F(6) − F(5)) = 5·(16 − 5)`; `F(11) = F(6)^2 + F(5)^2`.

**Bài 3**
1. Ở `i = 1` có `"a"` và `"b"`, cả hai run dài 1.
2. Dùng gán đồng thời `odd, even = ..., ...` để `even` nhận giá trị `odd` **cũ**.
3. `odd, even = 2, 0` rồi `for _ in range(n - 1): odd, even = (odd + even) % MOD, odd`.

**Bài 4**
1. Giống hệt `fast_pow` ở Mục 4.7, chỉ thay số bằng ma trận và `1` bằng ma trận đơn vị.
2. `while e: if e & 1: result = mat_mul(result, base); base = mat_mul(base, base); e >>= 1`.
3. `return 2 * mat_pow([[1, 1], [1, 0]], n - 1)[0][0] % MOD`.

**Bài 5**
1. `n` lẻ thì có thể không? Tổng các số chẵn luôn chẵn.
2. Chia đôi mọi độ dài run → một composition **bất kỳ** của `m = n / 2`. Số composition của `m` là `2^(m-1)` (mỗi "khe" trong `m − 1` khe giữa các đơn vị: cắt hoặc không).
3. `return 0 if n % 2 else 2 * pow(2, n // 2 - 1, MOD) % MOD` (tức `pow(2, n // 2, MOD)`).

**Bài 6**
1. Gọi `c(i)` = số composition của `i` với phần ∈ {1, 3}. Nhìn phần cuối: `c(i) = c(i-1) + c(i-3)`, với `c(0) = 1`, `c(-1) = c(-2) = 0`. Đáp án `2·c(n)`.
2. Vector `[c(i), c(i-1), c(i-2)]` → `[c(i+1), c(i), c(i-1)]` bằng ma trận `T = [[1, 0, 1], [1, 0, 0], [0, 1, 0]]`.
3. Bắt đầu từ `[c(0), c(-1), c(-2)] = [1, 0, 0]` nên `c(n) = (T^n)[0][0]`; viết `mat_mul` tổng quát bằng 3 vòng `for`.

# Đáp án tham khảo

Chỉ xem sau khi đã tự làm.

## Đáp án Bài 1 và Bài 2

**Bài 1:** runs: `aaabbb → [3, 3]` ✅, `aabaa → [2, 1, 2]` ❌, `ababa → [1, 1, 1, 1, 1]` ✅, `abbba → [1, 3, 1]` ✅, `abbbba → [1, 4, 1]` ❌, `bbbbb → [5]` ✅. `countGoodStrings(7) = 2·13 = 26`, `countGoodStrings(8) = 2·21 = 42`.

**Bài 2a:**

| i | 6 | 7 | 8 |
|---|---:|---:|---:|
| odd | 16 | 26 | 42 |
| even | 10 | 16 | 26 |

→ đáp án `n = 8` là **42**.

**Bài 2b:** `5 = 101₂` → `M^5 = M^1 · M^4 = [[1, 1], [1, 0]] · [[5, 3], [3, 2]] = [[8, 5], [5, 3]]`.

**Bài 2c:** `F(10) = 5·(16 − 5) = 55`, `F(11) = 64 + 25 = 89`.

In [41]:
# Bài 1
answer_1 = [is_good_runs(s) for s in bai1_strings]
assert answer_1 == [True, False, True, True, False, True]
assert sol.countGoodStrings(7) == 26 and sol.countGoodStrings(8) == 42

# Bài 2a
odd, even = 2, 0
for _ in range(7):
    odd, even = odd + even, odd
assert (odd, even) == (42, 26)

# Bài 2b
M = [[1, 1], [1, 0]]
M4 = mat_mul(mat_mul(M, M), mat_mul(M, M))
assert M4 == [[5, 3], [3, 2]]
assert mat_mul(M, M4) == [[8, 5], [5, 3]] == mat_pow(M, 5)

# Bài 2c
F5, F6 = 5, 8
assert F5 * (2 * F6 - F5) == 55 and F6**2 + F5**2 == 89 and fib_pair(10) == (55, 89)
print("Đáp án Bài 1, Bài 2 PASS:", answer_1)

Đáp án Bài 1, Bài 2 PASS: [True, False, True, True, False, True]


## Đáp án Bài 3

In [42]:
def answer_3_dp_linear(n: int) -> int:
    MOD = 10**9 + 7
    odd, even = 2, 0
    for _ in range(n - 1):
        odd, even = (odd + even) % MOD, odd
    return odd

assert answer_3_dp_linear(4) == 6
assert answer_3_dp_linear(3) == 4
assert answer_3_dp_linear(1) == 2
assert all(answer_3_dp_linear(n) == count_good_strings_bruteforce(n) for n in range(1, 11))
print("Đáp án Bài 3 PASS")

Đáp án Bài 3 PASS


## Đáp án Bài 4

In [43]:
def answer_4_matrix(n: int) -> int:
    MOD = 10**9 + 7

    def mul(A, B):
        return [[(A[i][0] * B[0][j] + A[i][1] * B[1][j]) % MOD for j in range(2)] for i in range(2)]

    def power(M, e):
        result = [[1, 0], [0, 1]]
        while e > 0:
            if e & 1:
                result = mul(result, M)
            M = mul(M, M)
            e >>= 1
        return result

    return 2 * power([[1, 1], [1, 0]], n - 1)[0][0] % MOD

assert answer_4_matrix(4) == 6
assert answer_4_matrix(1) == 2
assert answer_4_matrix(10**15) == Solution().countGoodStrings(10**15)
print("Đáp án Bài 4 PASS, n = 10^15 ->", answer_4_matrix(10**15))

Đáp án Bài 4 PASS, n = 10^15 -> 296650267


## Đáp án Bài 5

`n` lẻ → 0. `n = 2m` → chia đôi mọi độ dài run được một composition bất kỳ của `m`: có `2^(m-1)` cách; nhân 2 cho ký tự đầu → `2^m`.

In [44]:
def answer_5_even_runs(n: int) -> int:
    MOD = 10**9 + 7
    if n % 2 == 1:
        return 0
    return pow(2, n // 2, MOD)

def brute_even_runs(n: int) -> int:
    return sum(1 for t in product("ab", repeat=n) if all(L % 2 == 0 for L in run_lengths(t)))

assert answer_5_even_runs(3) == 0 and answer_5_even_runs(4) == 4 and answer_5_even_runs(2) == 2
assert all(answer_5_even_runs(n) == brute_even_runs(n) for n in range(1, 13))
print("Đáp án Bài 5 PASS:", [answer_5_even_runs(n) for n in range(1, 13)])
print("n = 10^15 ->", answer_5_even_runs(10**15))

Đáp án Bài 5 PASS: [0, 2, 0, 4, 0, 8, 0, 16, 0, 32, 0, 64]
n = 10^15 -> 949159125


## Đáp án Bài 6

`c(i) = c(i-1) + c(i-3)`; ma trận `T = [[1, 0, 1], [1, 0, 0], [0, 1, 0]]`; đáp án `2·(T^n)[0][0]`.

In [45]:
def answer_6_runs_1_or_3(n: int) -> int:
    MOD = 10**9 + 7

    def mul(A, B):
        size = len(A)
        return [[sum(A[i][t] * B[t][j] for t in range(size)) % MOD for j in range(size)]
                for i in range(size)]

    def power(M, e):
        size = len(M)
        result = [[int(i == j) for j in range(size)] for i in range(size)]
        while e > 0:
            if e & 1:
                result = mul(result, M)
            M = mul(M, M)
            e >>= 1
        return result

    T = [[1, 0, 1],
         [1, 0, 0],
         [0, 1, 0]]
    return 2 * power(T, n)[0][0] % MOD

def brute_runs_1_or_3(n: int) -> int:
    return sum(1 for t in product("ab", repeat=n) if all(L in (1, 3) for L in run_lengths(t)))

assert [answer_6_runs_1_or_3(n) for n in range(1, 9)] == [2, 2, 4, 6, 8, 12, 18, 26]
assert all(answer_6_runs_1_or_3(n) == brute_runs_1_or_3(n) for n in range(1, 13))
print("Đáp án Bài 6 PASS, n = 10^15 ->", answer_6_runs_1_or_3(10**15))

Đáp án Bài 6 PASS, n = 10^15 -> 368808168


# Tổng kết

## Con đường suy luận của bài

```text
Định nghĩa đệ quy "chuỗi tốt"
   │  brute force + thí nghiệm n nhỏ
   ▼
Tốt  ⇔  mọi run có độ dài lẻ                     (Bước đột phá 1)
   │  chuỗi = (ký tự đầu, độ dài các run)
   ▼
Đáp án = 2 × (số composition lẻ của n)           (Bước đột phá 2)
   │  DP theo phần cuối → nhìn dãy số
   ▼
Số composition lẻ của n = F(n)                   (Bước đột phá 3)
   │  DP trạng thái = nhân ma trận [[1,1],[1,0]]
   ▼
Lũy thừa ma trận / fast doubling: O(log n)
```

## Template nên nhớ — fast doubling dạng vòng lặp

```python
class Solution:
    def countGoodStrings(self, n: int) -> int:
        MOD = 10**9 + 7
        a, b = 0, 1                              # (F(k), F(k+1)), k = 0
        for bit in bin(n)[2:]:
            c = a * ((2 * b - a) % MOD) % MOD    # F(2k)
            d = (a * a + b * b) % MOD            # F(2k+1)
            a, b = (d, (c + d) % MOD) if bit == "1" else (c, d)
        return 2 * a % MOD
```

## Template lũy thừa ma trận (dùng cho mọi truy hồi tuyến tính)

```python
def mat_pow(M, e):
    result = identity
    while e > 0:
        if e & 1:
            result = mat_mul(result, M)
        M = mat_mul(M, M)
        e >>= 1
    return result
```

## Ba câu cần nhớ

> 1. **Định nghĩa đệ quy rối → brute force → in kết quả n nhỏ → tìm đặc trưng đơn giản.**
> 2. **Điều kiện trên các run → DP trạng thái (chẵn/lẻ của run hiện tại).**
> 3. **DP tuyến tính + n khổng lồ → ma trận + lũy thừa nhanh, nhớ `% MOD`.**

# Checklist trước khi submit

- [ ] Đã **nhân 2** cho hai lựa chọn ký tự đầu (`'a'` hoặc `'b'`) chưa?
- [ ] Thử lại `n = 1, 2, 3, 4` có ra `2, 2, 4, 6` không? (bắt lỗi lệch chỉ số Fibonacci)
- [ ] Thuật toán là `O(log n)`, **không** có vòng lặp chạy tới `n`?
- [ ] Đã lấy `% MOD` sau **mỗi** phép nhân / cộng?
- [ ] **Không** dùng `float` / công thức Binet?
- [ ] Nếu dùng đệ quy: độ sâu chỉ `O(log n)`, không phải `O(n)`?
- [ ] Signature đúng: `def countGoodStrings(self, n: int) -> int` (đối chiếu tên hàm với editor LeetCode)?
- [ ] Nếu viết bằng C++/Java: dùng `long long` / `long`, và cộng `MOD` trước khi lấy dư ở biểu thức `2*b - a`?